<a href="https://colab.research.google.com/github/ronyates47/Gedcom-Utils/blob/main/anchor_engine_v7_base_runall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# @title [CELL 1] Environment Setup & Configuration
print("="*60)
print("      [CELL 1] SYSTEM WAKE-UP & ENVIRONMENT SETUP...")
print("="*60)

# 1. Install required packages (ensures Colab has what we need)
!pip install -q pandas pytz

# 2. Import standard libraries used across the entire pipeline
import os
import re
import csv
import json
import math
import shutil
import pandas as pd
import pytz
from datetime import datetime
from ftplib import FTP_TLS

# 3. Securely load FTP Credentials from Colab Secrets
try:
    from google.colab import userdata
    print("\n[+] Google Colab Environment Detected. Loading secrets...")

    # Check and set FTP Host
    HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
    if HOST:
        os.environ["FTP_HOST"] = HOST
        print("    ✅ FTP_HOST loaded.")
    else:
        print("    ⚠️ WARNING: FTP_HOST not found in secrets.")

    # Check and set FTP User
    USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
    if USER:
        os.environ["FTP_USER"] = USER
        print("    ✅ FTP_USER loaded.")
    else:
        print("    ⚠️ WARNING: FTP_USER not found in secrets.")

    # Check and set FTP Password
    PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
    if PASS:
        os.environ["FTP_PASS"] = PASS
        print("    ✅ FTP_PASS loaded.")
    else:
        print("    ⚠️ WARNING: FTP_PASS not found in secrets.")

except ImportError:
    print("\n[!] Running outside of Google Colab. Make sure environment variables are set manually.")
except userdata.SecretNotFoundError as e:
    print(f"\n❌ SECRET ERROR: {e}")
    print("    Please click the '🔑 Keys' icon on the left sidebar and ensure FTP_HOST, FTP_USER, and FTP_PASS are saved.")

print("\n✅ Cell 1 (Environment Setup) Complete. The system is ready.")
print("="*60)

      [CELL 1] SYSTEM WAKE-UP & ENVIRONMENT SETUP...

[+] Google Colab Environment Detected. Loading secrets...
    ✅ FTP_HOST loaded.
    ✅ FTP_USER loaded.
    ✅ FTP_PASS loaded.

✅ Cell 1 (Environment Setup) Complete. The system is ready.


In [20]:
# @title [CELL 2] The Data Interceptor (Smart GEDCOM Fetch)
import os
import glob
from ftplib import FTP_TLS

print("="*60)
print("      📥 PRE-LOADING DATA (SMART FETCH)...")
print("="*60)

# 1. LOCAL CACHE CHECK: Do we already have a GEDCOM?
local_geds = glob.glob("*.ged")
gedcom_needed = True

if local_geds:
    print(f"[+] Local cache found: {local_geds[0]}. Skipping FTP download for GEDCOM.")
    gedcom_needed = False
    gedcom_name = local_geds[0]

try:
    # Uses the secrets already loaded by Cell 1
    ftps = FTP_TLS()
    ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()

    # 2. FETCH NEWEST GEDCOM FROM TNG
    if gedcom_needed:
        print("[+] Searching for the newest GEDCOM in /tng/gedcom/...")

        # Navigate to the TNG folder from the root login directory
        try:
            ftps.cwd("/tng/gedcom")
        except:
            ftps.cwd("tng/gedcom") # Fallback for relative path

        files = ftps.nlst()
        ged_files = [f for f in files if f.lower().endswith('.ged')]

        if not ged_files:
            print("    ❌ ERROR: No .ged files found in /tng/gedcom/")
        else:
            newest_ged = None
            newest_time = ""

            # Ask the server for the exact modification time of every file
            for ged in ged_files:
                try:
                    res = ftps.voidcmd(f"MDTM {ged}")
                    mtime = res[4:].strip() # Returns format like 20260302123000
                    if mtime > newest_time:
                        newest_time = mtime
                        newest_ged = ged
                except:
                    pass # Skip if server denies MDTM for a specific file

            # Fallback to alphabetical sorting if server time-check fails
            if not newest_ged:
                newest_ged = sorted(ged_files)[-1]

            print(f"    [+] Newest GEDCOM identified: {newest_ged}")
            print(f"    [+] Downloading {newest_ged} to Colab memory...")

            with open(newest_ged, "wb") as f:
                ftps.retrbinary(f"RETR {newest_ged}", f.write)
            print("    ✅ GEDCOM successfully loaded.")

    # 3. FETCH CSV FROM THE ANCHOR SPOKE
    print("\n[+] Downloading engine_database.csv from Anchor Spoke...")

    # Jump back to the root directory, then into the Anchor folder
    ftps.cwd("/")
    try:
        ftps.cwd("anchor/anc-1000-yates")
    except:
        ftps.cwd("/anchor/anc-1000-yates")

    with open("engine_database.csv", "wb") as f:
        ftps.retrbinary("RETR engine_database.csv", f.write)
    print("    ✅ CSV successfully loaded.")

    ftps.quit()
    print("\n🎉 Smart Fetch Complete! Cell 2 is ready to run.")

except Exception as e:
    print(f"\n❌ FTP Download Failed: {e}")

      📥 PRE-LOADING DATA (SMART FETCH)...
[+] Local cache found: yates_study_2025.ged. Skipping FTP download for GEDCOM.

[+] Downloading engine_database.csv from Anchor Spoke...
    ✅ CSV successfully loaded.

🎉 Smart Fetch Complete! Cell 2 is ready to run.


In [21]:
# @title [CELL 3] The Data & Math Engine (Zero-Match Injection Update)
print("="*60)
print("      [CELL 2] DATA & MATH ENGINE STARTING...")
print("="*60)

import os, sys, re, csv, json, math, shutil
import pandas as pd
from ftplib import FTP_TLS
try:
    from google.colab import userdata
except ImportError:
    pass

CSV_DB = "engine_database.csv"
JSON_DB = "compiled_database.json"
KEY_FILE = "match_to_unmasked.csv"
PROCESSED_GED = "_processed_unmasked.ged"

try:
    HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
    USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
    PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
except: pass

# --- STEP 1: FETCH FILES ---
if not os.path.exists(KEY_FILE):
    print(f"    [+] Fetching {KEY_FILE} via FTP...")
    try:
        ftps = FTP_TLS(); ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()
        with open(KEY_FILE, "wb") as f: ftps.retrbinary(f"RETR /anchor/anc-1000-yates/{KEY_FILE}", f.write)
        ftps.quit()
    except Exception as e: print(f"    ⚠️ FTP fetch failed: {e}")

ged_files = [f for f in os.listdir('.') if f.lower().endswith('.ged') and "_processed" not in f.lower()]
if not ged_files:
    print("❌ No GEDCOM found. Please upload one.")
else:
    DEFAULT_GEDCOM = sorted(ged_files, key=lambda x: os.path.getmtime(x), reverse=True)[0]
    shutil.copyfile(DEFAULT_GEDCOM, PROCESSED_GED)
    print(f"    [+] Active GEDCOM: {DEFAULT_GEDCOM}")

# --- STEP 2: LOAD AUTH & PARSE GEDCOM ---
csv_auth = {}
if os.path.exists(KEY_FILE):
    with open(KEY_FILE, 'r', errors='replace') as f:
        for i, row in enumerate(csv.reader(f)):
            if i > 0 and len(row) >= 2:
                code, name = row[0].strip().lower(), row[1].strip()
                tid = "I" + re.sub(r'[^0-9]', '', row[2]) if len(row)>2 and row[2].strip() else ""
                sort_key = row[3].strip().lower() if len(row)>3 else ""
                csv_auth[code] = {"name": name, "id": tid, "sort_key": sort_key}

individuals = {}; families = {}; persons_data = {}
current_id = None; current_fam = None; mode = None

def clean_name(n):
    if not n: return "findme"
    s = n.replace("/", "").strip()
    if s.lower() in ["unknown", "missing", "searching", "living", "private", "nee", "wife"] or "?" in s: return "findme"
    return s

with open(PROCESSED_GED, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip(); parts = line.split(" ", 2)
        if len(parts) < 2: continue
        lvl, tag, val = parts[0], parts[1], parts[2] if len(parts)>2 else ""

        if lvl == "0" and "INDI" in val:
            current_id = tag.replace("@", "")
            individuals[current_id] = {"name": "findme", "famc": None, "fams": [], "code": "", "cm": 0}
            persons_data[current_id] = {'name': f'ID {current_id}', 'bdate': '', 'bplace': '', 'ddate': '', 'dplace': '', 'sources_count': 0, 'citations_count': 0}
            current_fam = None; mode = None
        elif current_id and lvl != "0":
            if tag == "NAME" and lvl == "1":
                individuals[current_id]["name"] = clean_name(val)
                persons_data[current_id]["name"] = val.replace('/', '').strip()
            elif tag == "FAMC" and lvl == "1": individuals[current_id]["famc"] = val.replace("@", "")
            elif tag == "FAMS" and lvl == "1": individuals[current_id]["fams"].append(val.replace("@", ""))
            elif tag == "NPFX" and lvl == "2":
                m_code = re.search(r'(\d+)\s*&?\s*([^ \t\n\r\f\v]+)', val)
                if m_code: individuals[current_id]["code"] = m_code.group(2).lower()
                m_cm = re.search(r'^(\d+)|(\d+)\s*cM', val, re.IGNORECASE)
                if m_cm: individuals[current_id]["cm"] = int(m_cm.group(1) or m_cm.group(2))
            elif tag == "BIRT": mode = "BIRT"
            elif tag == "DEAT": mode = "DEAT"
            elif tag == "SOUR" and lvl == "1": persons_data[current_id]['sources_count'] += 1; mode = None
            elif tag == "SOUR" and lvl == "2": persons_data[current_id]['citations_count'] += 1
            elif tag == "DATE" and mode == "BIRT": persons_data[current_id]['bdate'] = val
            elif tag == "DATE" and mode == "DEAT": persons_data[current_id]['ddate'] = val
            elif tag == "PLAC" and mode == "BIRT": persons_data[current_id]['bplace'] = val
            elif tag == "PLAC" and mode == "DEAT": persons_data[current_id]['dplace'] = val
            elif lvl == "1": mode = None

        elif lvl == "0" and "FAM" in val:
            current_fam = tag.replace("@", "")
            families[current_fam] = {"husb": None, "wife": None}
            current_id = None
        elif current_fam and lvl != "0":
            if tag == "HUSB": families[current_fam]["husb"] = val.replace("@", "")
            elif tag == "WIFE": families[current_fam]["wife"] = val.replace("@", "")

# --- STEP 3: BUILD BASE CSV ROWS ---
print("    [+] Tracing Lineages & Generating CSV...")
def get_parents(pid):
    if not pid or pid not in individuals: return None, None
    famc = individuals[pid]["famc"]
    if not famc or famc not in families: return None, None
    return families[famc]["husb"], families[famc]["wife"]

yates_memo = {}
def has_yates(pid):
    if not pid or pid not in individuals: return False
    if pid in yates_memo: return yates_memo[pid]
    n = individuals[pid]["name"].lower()
    if "yates" in n or "yeates" in n: yates_memo[pid] = True; return True
    d, m = get_parents(pid)
    res = has_yates(d) or has_yates(m)
    yates_memo[pid] = res; return res

def climb(start_id):
    curr = start_id; lin = []
    while curr:
        p = individuals.get(curr)
        if not p: break
        spouse_name = "findme"; sp_id = None
        if p["fams"]:
            fid = p["fams"][0]
            if fid in families:
                f = families[fid]
                sp_id = f["wife"] if f["husb"] == curr else f["husb"]
                if sp_id and sp_id in individuals: spouse_name = individuals[sp_id]["name"]
        lin.append({"name": p["name"], "id": curr, "spouse": spouse_name, "sp_id": sp_id})
        d, m = get_parents(curr)
        if not d and not m: break
        dy, my = has_yates(d), has_yates(m)
        curr = d if dy and not my else (m if my and not dy else (d if d else m))
    return lin

for code, td in csv_auth.items():
    if td["id"] and td["id"] in individuals:
        full = list(reversed(climb(td["id"])))
        td["lin_str"] = " -> ".join([x["name"] for x in full])
        td["pids"] = ",".join([x["id"] for x in full])
    else:
        td["lin_str"] = ""; td["pids"] = ""

rows = []
for uid, p in individuals.items():
    if p["code"]:
        kc = p["code"]
        td = csv_auth.get(kc, {"name": kc, "id": "", "sort_key": "", "lin_str": "", "pids": ""})
        t_disp = f"{td['name']} [{td['id']}]" if td['id'] else f"{td['name']} [{kc}]"

        lin = climb(uid)
        if not lin: continue
        full = list(reversed(lin))

        gen1 = full[0]
        top_name = gen1["name"]; sp_name = gen1["spouse"]
        pair_simp = f"{top_name} & {sp_name}" if sp_name != "findme" else top_name

        sur = top_name.split()[-1] if top_name.split() else ""
        firsts = re.sub(f"{re.escape(sur)}$", "", top_name).strip() if sur else top_name
        b_yr = re.search(r'\d{4}', persons_data[gen1['id']]['bdate'])
        d_yr = re.search(r'\d{4}', persons_data[gen1['id']]['ddate'])
        dates = f"({b_yr.group(0) if b_yr else 'findme'} - {d_yr.group(0) if d_yr else 'findme'})"
        dir_lbl = f"{sur}, {firsts} {dates}" + (f" & {sp_name}" if sp_name != "findme" else "")

        rows.append({
            "Tester_Code": kc, "Tester_Name": td["name"], "Tester_ID": td["id"], "Tester_Display": t_disp,
            "Tester_Sort_Key": td["sort_key"], "Tester_Lineage": td["lin_str"], "Tester_Path_IDs": td["pids"],
            "Match_Name": p["name"], "Match_ID": uid, "cM": p["cm"],
            "Match_Lineage": " -> ".join([pair_simp] + [x["name"] for x in full[1:]]),
            "Match_Path_IDs": ",".join([x["id"] for x in full]),
            "Authority_Directory_Label": dir_lbl
        })

df = pd.DataFrame(rows)
df.to_csv(CSV_DB, index=False, encoding="iso-8859-15", quoting=csv.QUOTE_ALL)
df.rename(columns={"Authority_Directory_Label": "Dir_Label", "Tester_Code": "Kit_Code", "Match_Lineage": "Lineage", "Match_Path_IDs": "s_ids", "Tester_Display": "Kit_Name"}, inplace=True)
df['search_ids'] = df['s_ids']; df['search_names'] = df['Lineage'].astype(str).str.replace(' -> ', '|')
df['t_names'] = df['Tester_Lineage'].astype(str).str.replace(' -> ', '|'); df['t_ids'] = df['Tester_Path_IDs'].astype(str).str.replace(',', '|')

# --- STEP 4: MATHEMATICAL ENGINE (ANCHOR/CSS) ---
print("    [+] Calculating Math Matrices & Audits...")
def clean_num(s): return re.sub(r'[^0-9]', '', str(s))
def norm_log(val, cap): return min(1.0, math.log(1 + max(0, val or 0)) / math.log(1 + cap))

id_to_kits = {}
for _, r in df.iterrows():
    for i in [clean_num(x) for x in str(r['s_ids']).split(',') if x]:
        id_to_kits.setdefault(i, set()).add(r['Kit_Name'])

participant_scores = []
for p_name, grp in df.groupby('Kit_Name'):
    val_grp = grp[grp['Dir_Label'] != 'No Matches']
    pm = len(val_grp)
    if pm == 0: continue

    dc = val_grp['Dir_Label'].value_counts()
    hc_t, hc_2 = (int(dc.iloc[0]), int(dc.iloc[1])) if len(dc)>1 else (int(dc.iloc[0]), 0) if len(dc)>0 else (0,0)
    target_anc = str(dc.index[0]) if len(dc)>0 else "Unknown"

    hh, target_id = 0, None
    for _, r in val_grp.iterrows():
        for i in [clean_num(x) for x in str(r['s_ids']).split(',') if x]:
            if len(id_to_kits.get(i, set())) > hh: hh, target_id = len(id_to_kits[i]), i

    spine_ids = []; br = 0; tb = 0
    if target_id:
        tb = len(id_to_kits.get(target_id, set()))
        col = df[df['s_ids'].apply(lambda x: target_id in [clean_num(y) for y in str(x).split(',') if y])]
        b_set = set()
        for _, r in col.iterrows():
            ids = [clean_num(x) for x in str(r['s_ids']).split(',') if x]
            nms = str(r['search_names']).split('|')
            if target_id in ids:
                idx = ids.index(target_id)
                b_set.add(nms[idx+1].replace('findme', '?').replace('FINDME', '?').split(' (')[0].strip() if idx+1 < len(nms) else "Direct")
        br = len(b_set)
        spine_ids = [clean_num(x) for x in str(val_grp[val_grp['s_ids'].apply(lambda x: target_id in str(x))].iloc[0]['s_ids']).split(',') if x]

    # Math formulas
    dr = float(hc_t / max(1, hc_2))
    w_sum = norm_log(pm, 150) + norm_log(hc_t, 100) + (norm_log(dr, 10)*1.5) + norm_log(tb, 40) + norm_log(hh, 150)
    w_sum += 2.0 * (1.0 if br>=6 else 0.85 if br==5 else 0.70 if br==4 else 0.50 if br==3 else 0.25 if br==2 else 0)
    st_str, st_val = ("PASS", 1.0) if pm>=15 and br>=3 and dr>=1.5 else ("PARTIAL", 0.85) if pm>=15 and br>=2 else ("FAIL", 0.6)
    css = float(100 * (w_sum / 7.5) * st_val)

    # Doc Audit
    tp_viol = 0; tp_checks = 0; diag_log = []
    for pid in spine_ids:
        d = persons_data.get(pid, {})
        if d.get('sources_count',0) + d.get('citations_count',0) == 0: diag_log.append(f"Missing Sources: {d.get('name')} (I{pid})")

    docs_score = 88.5 if len(diag_log)==0 else max(35.0, 85.0 - (len(diag_log)*5))
    anchor = min(100.0, (0.65 * css) + (0.35 * docs_score) + (10.0 * min(css, docs_score)/100.0))

    participant_scores.append({'pName': str(p_name), 'targetAnc': target_anc, 'targetID': str(target_id or ''), 'PM': int(pm), 'HC_T': int(hc_t), 'DR': float(dr), 'BR': int(br), 'NS': int(hh), 'ST_str': st_str, 'cssFinal': float(css), 'DOCS': float(docs_score), 'ANCHOR': float(anchor), 'diagLog': diag_log, 'AX': 1, 'CC': 1.0, 'BC': 1.0, 'DC': 1.0, 'TP': 1.0, 'isGroup': False, 'GD': len(spine_ids)})

anc_data = {}; part_data = {}
for lbl, grp in df.groupby('Dir_Label'):
    if str(lbl).strip() == 'No Matches' or not str(lbl).strip(): continue
    u_t = len(grp['Kit_Name'].unique())
    anc_data[str(lbl)] = {"name": str(lbl), "matches": len(grp), "cm": int(grp['cM'].sum()), "badge": "Platinum" if len(grp)>=30 else "Gold", "integrity": min(100, (len(grp)*2) + (u_t*10)), "testers": u_t}

for kname, grp in df.groupby('Kit_Name'):
    val_grp = grp[grp['Dir_Label'] != 'No Matches']
    p_score = next((i for i in participant_scores if i["pName"] == str(kname)), None)
    part_data[str(kname)] = {
        "name": str(kname), "sort_key": str(grp.iloc[0].get('Tester_Sort_Key', str(kname).lower())).strip(),
        "matches": len(val_grp), "cm": int(val_grp['cM'].sum()) if not val_grp.empty else 0,
        "badge": "Keystone Tester" if len(val_grp)>=15 else "Study Participant",
        "css_status": p_score['ST_str'] if p_score else "UNSCORED", "ns": int(p_score['NS']) if p_score else 0,
        "br": int(p_score['BR']) if p_score else 0, "target_id": p_score['targetID'] if p_score else '',
        "docs_score": float(p_score['DOCS']) if p_score else 0.0, "diag_log": p_score['diagLog'] if p_score else [],
        "kit_code": str(grp.iloc[0]['Kit_Code']), "integrity": 95
    }

# ✨ THE FIX: INJECTING THE ZERO-MATCH PARTICIPANTS INTO THE SYSTEM!
for code, td in csv_auth.items():
    t_disp = f"{td['name']} [{td['id']}]" if td['id'] else f"{td['name']} [{code}]"
    if t_disp not in part_data:
        part_data[t_disp] = {
            "name": t_disp, "sort_key": td["sort_key"] or td["name"].lower(),
            "matches": 0, "cm": 0, "badge": "Study Participant",
            "css_status": "UNSCORED", "ns": 0, "br": 0, "target_id": "",
            "docs_score": 0.0, "diag_log": ["No correlated matches found in GEDCOM."],
            "kit_code": code, "integrity": 0
        }

# --- STEP 5: SAVE GLOBALS TO DISK ---
print("    [+] Saving Compiled JS_GLOBALS to Disk...")
db_json = df[['Dir_Label', 'Kit_Name', 'cM', 'Match_ID', 'Lineage', 'search_ids', 'search_names', 't_names', 't_ids', 'Tester_ID', 'Kit_Code']].rename(columns={'Dir_Label':'ancestor', 'Kit_Name':'participant', 'cM':'cm', 'Match_ID':'id', 'Lineage':'lineage', 'Tester_ID':'tester_id', 'Kit_Code':'kit_code'}).to_dict(orient='records')
final_json = {"PRECOMPUTED": participant_scores, "DATA": {"ancestors": anc_data, "participants": part_data, "persons": persons_data}, "DB": db_json}

def safe_json_encoder(obj):
    if pd.isna(obj): return None
    if hasattr(obj, 'item'): return obj.item()
    return str(obj)

with open(JSON_DB, "w") as f:
    json.dump(final_json, f, default=safe_json_encoder)

print(f"\n[SUCCESS] Cell 2 Complete. Data successfully crunched and saved to {JSON_DB}.")

      [CELL 2] DATA & MATH ENGINE STARTING...
    [+] Active GEDCOM: yates_study_2025.ged
    [+] Tracing Lineages & Generating CSV...
    [+] Calculating Math Matrices & Audits...
    [+] Saving Compiled JS_GLOBALS to Disk...

[SUCCESS] Cell 2 Complete. Data successfully crunched and saved to compiled_database.json.


In [22]:
# @title [CELL 4] The Vault Fetcher (Hub Integration)
import urllib.request

print("="*60)
print("      📥 DOWNLOADING VAULT FRAGMENTS FROM HUB...")
print("="*60)

# The 22 fragments we just stored in London
fragment_keys = [
    'css_base', 'nav', 'site_info', 'js_core', 'btt_btn', 'proof_engine_tmpl',
    'dossier_tmpl', 'anchor_content', 'anchor_theory', 'anchor_css', 'network_page',
    'consolidator_html', 'consolidator_js', 'consolidator_css', 'admin_css',
    'register_css', 'glossary_inline', 'gedmatch_inline', 'contents_content',
    'share_content', 'subscribe_content', 'theory_page'
]

SITE_TEMPLATES = {}
hub_url = "https://yates.one-name.net/anchor/anchor-hub/"

print("[+] Reaching into London Hub for text blocks...")

# Disguise as a browser to bypass the 403 Firewall
req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}

for key in fragment_keys:
    try:
        req = urllib.request.Request(f"{hub_url}{key}.txt", headers=req_headers)
        SITE_TEMPLATES[key] = urllib.request.urlopen(req).read().decode('utf-8')
    except Exception as e:
        print(f"    [-] Missing Fragment: {key}.txt ({e})")
        SITE_TEMPLATES[key] = f""

print(f"    ✅ Successfully loaded {len(SITE_TEMPLATES)} fragments into Engine Memory!")
print("\n🎉 Vault Fetcher Complete. Ready for Publisher.")

      📥 DOWNLOADING VAULT FRAGMENTS FROM HUB...
[+] Reaching into London Hub for text blocks...
    ✅ Successfully loaded 22 fragments into Engine Memory!

🎉 Vault Fetcher Complete. Ready for Publisher.


In [26]:
# @title [CELL 5A] UI & Content Payloads (Bulletproof DOM Safety & Precision Math)
print("="*60)
print("      [CELL 5A] LOADING UI PAYLOADS INTO MEMORY...")
print("="*60)

ANCHOR_PAYLOADS = {}

ANCHOR_PAYLOADS['table_cleanup_css'] = """
<style>
.table-scroll-wrapper { width: 100%; display: flex; justify-content: center; padding: 20px 15px; box-sizing: border-box; }
table#reg-table { width: 100% !important; max-width: 1100px !important; margin: 0 auto !important; border-collapse: collapse !important; background: #ffffff !important; box-shadow: 0 4px 15px rgba(0,0,0,0.08) !important; border-radius: 8px !important; overflow: hidden !important; border: hidden !important; }
table#reg-table thead th { background: #004d40 !important; color: #ffffff !important; padding: 18px !important; text-align: center !important; font-size: 16px !important; font-weight: bold !important; border: none !important; letter-spacing: 1px !important; text-transform: uppercase; }
table#reg-table tbody td { padding: 16px 24px !important; border-bottom: 1px solid #eeeeee !important; border-top: none !important; border-left: none !important; border-right: none !important; text-align: left !important; font-size: 15px !important; color: #333333 !important; }
table#reg-table tbody tr:last-child td { border-bottom: none !important; }
table#reg-table tbody tr:hover td { background-color: #f1f8e9 !important; }
</style>
"""

ANCHOR_PAYLOADS['warning_banner'] = """
<style>
.scrolling-banner { background-color: #d32f2f; color: #ffffff; padding: 10px 0; font-family: sans-serif; font-weight: bold; font-size: 16px; letter-spacing: 1px; overflow: hidden; white-space: nowrap; position: relative; width: 100%; z-index: 99999; border-bottom: 3px solid #b71c1c; box-shadow: 0 2px 5px rgba(0,0,0,0.2); }
.scrolling-text { display: inline-block; padding-left: 100%; animation: scroll-banner 25s linear infinite; }
@keyframes scroll-banner { 0% { transform: translate(0, 0); } 100% { transform: translate(-100%, 0); } }
.scrolling-banner:hover .scrolling-text { animation-play-state: paused; cursor: default; }
</style>
<div class="scrolling-banner no-print">
    <div class="scrolling-text">
        ⚠️ THIS STUDY VERSION HAS NOT BEEN RELEASED TO THE PUBLIC. WELCOME TO BROWSE AND MAKE SUGGESTIONS, BUT DO NOT SHARE OR MATERIALLY USE UNTIL THIS BANNER IS REMOVED. ⚠️
    </div>
</div>
"""

ANCHOR_PAYLOADS['velvet_rope'] = """
<div id="velvet-rope" style="position:fixed; top:0; left:0; width:100%; height:100%; background:#eceff1; z-index:99999; display:flex; flex-direction:column; align-items:center; justify-content:center; font-family:sans-serif;">
    <div style="background:white; padding:40px; border-radius:8px; box-shadow:0 4px 15px rgba(0,0,0,0.1); text-align:center; border-top: 5px solid #006064;">
        <h2 style="color:#004d40; margin-top:0;">Admin Access Required</h2>
        <p style="color:#666; margin-bottom:20px;">This area is for authorized researchers.</p>
        <input type="password" id="vr-pass" placeholder="Passphrase" style="padding:10px; font-size:16px; border:2px solid #ddd; border-radius:4px; margin-bottom:15px; width:200px; text-align:center; outline:none;" onkeypress="if(event.key === 'Enter') checkVR()">
        <br>
        <button onclick="checkVR()" style="padding:10px 25px; font-size:16px; background:#00838f; color:white; border:none; border-radius:4px; cursor:pointer; font-weight:bold;">Unlock</button>
        <p id="vr-err" style="color:#c62828; display:none; margin-top:15px; font-size:14px; font-weight:bold;">Incorrect passphrase.</p>
    </div>
</div>
<script>
if(sessionStorage.getItem('admin_unlocked') === 'yes') { document.getElementById('velvet-rope').style.display = 'none'; }
function checkVR() {
    if(document.getElementById('vr-pass').value === 'ons') {
        sessionStorage.setItem('admin_unlocked', 'yes');
        document.getElementById('velvet-rope').style.display = 'none';
    } else {
        document.getElementById('vr-err').style.display = 'block';
        document.getElementById('vr-pass').value = '';
    }
}
</script>
"""

# 🌟 GLOBAL SCRIPTS & UNIVERSAL MATRIX UPGRADER 🌟
ANCHOR_PAYLOADS['deep_path_js'] = """
<script>
document.addEventListener('DOMContentLoaded', function() {

    // Hide Old "Establishing Kinship" Box on Registers
    document.querySelectorAll('*').forEach(el => {
        if(el.children.length === 0 && (el.textContent.includes('Establishing Kinship Through Collateral DNA Saturation') || el.textContent.includes('This register employs Collateral DNA Saturation'))) {
            let container = el.closest('div[style*="background"], fieldset, .alert');
            if(container) container.style.display = 'none';
            else el.style.display = 'none';
        }
    });

    function upgradePresets() {
        document.querySelectorAll('select').forEach(sel => {
            let ident = (sel.id + " " + sel.className).toLowerCase();
            if(ident.includes('preset') || ident.includes('participant') || ident.includes('filter')) {
                Array.from(sel.options).forEach(opt => { if(opt.text.includes('All Fails')) opt.remove(); });
                if(sel.dataset.upgraded) return;
                sel.dataset.upgraded = 'true';
                let hasAll = Array.from(sel.options).some(o => o.value === 'ALL_OVERRIDE');
                if(!hasAll) {
                    let opt = document.createElement('option');
                    opt.value = 'ALL_OVERRIDE';
                    opt.textContent = '🌟 All Participants';
                    opt.style.fontWeight = 'bold';
                    opt.style.color = '#004d40';
                    opt.style.background = '#e0f2f1';
                    if(sel.options.length > 0 && (sel.options[0].text.includes('Select') || sel.options[0].text.includes('Choose'))) {
                        sel.insertBefore(opt, sel.options[1]);
                    } else { sel.insertBefore(opt, sel.options[0]); }
                }
            }
        });
    }

    document.addEventListener('change', function(e) {
        if (e.target && e.target.tagName === 'SELECT') {
            let isAll = e.target.multiple ? Array.from(e.target.selectedOptions).some(o => o.value === 'ALL_OVERRIDE') : (e.target.value === 'ALL_OVERRIDE');
            if (isAll) {
                let changed = false;
                let checkboxes = document.querySelectorAll('input[type="checkbox"]');
                if (checkboxes.length > 0) {
                    checkboxes.forEach(cb => {
                        if (!cb.checked && cb.value !== 'all' && cb.name !== 'ignore') { cb.checked = true; changed = true; }
                    });
                    let customTitleInput = document.querySelector('input[type="text"][placeholder*="e.g."]');
                    if(customTitleInput) customTitleInput.value = "All Participants Report";
                } else {
                    let targetSel = e.target;
                    let ident = (e.target.id + " " + e.target.className).toLowerCase();
                    if (ident.includes('preset')) {
                        let ms = document.querySelector('select[multiple]');
                        if(ms) targetSel = ms;
                    }
                    Array.from(targetSel.options).forEach(o => {
                        if (o.value && o.value !== 'ALL_OVERRIDE' && o.value !== 'none' && !o.text.includes('Select') && !o.text.includes('Choose')) {
                            o.selected = true; changed = true;
                        }
                    });
                    if (e.target.multiple) {
                        Array.from(e.target.options).forEach(o => { if(o.value === 'ALL_OVERRIDE') o.selected = false; });
                    } else if (e.target !== targetSel) { e.target.selectedIndex = 0; }
                    if (changed) targetSel.dispatchEvent(new Event('change', { bubbles: true }));
                }
            }
        }
    });

    function applyUniversalMatrixUpgrades() {
        let t_pm = parseInt(localStorage.getItem('ANC_CFG_PM')) || 15;
        let t_br = parseInt(localStorage.getItem('ANC_CFG_COL')) || 2;
        let t_vol = parseFloat(localStorage.getItem('ANC_CFG_VOL')) || 40.0;
        let t_dr = 3.0;

        document.querySelectorAll('table').forEach(table => {
            let ths = Array.from(table.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            if (!ths.includes('PM') && !ths.includes('STATUS') && !ths.includes('LVS')) return;

            let pIdx = ths.indexOf('PM');
            let bIdx = ths.indexOf('BR');
            let drIdx = ths.indexOf('DR');
            let vIdx = ths.findIndex(th => th.includes('VOL') || th.includes('CM'));
            let stIdx = ths.findIndex(th => th === 'STATUS' || th === 'ST' || th === 'LVS' || th.includes('VALID'));
            let ancIdx = ths.findIndex(th => th.includes('ANCESTOR') || th.includes('LINEAGE') || th.includes('NODE'));
            let docsIdx = ths.findIndex(th => th === 'DOCS' || th.includes('PAPER') || th.includes('SCORE') && !th.includes('CSS') && !th.includes('ANCHOR'));

            if (stIdx > -1 && table.querySelectorAll('th')[stIdx] && !table.querySelectorAll('th')[stIdx].innerText.includes('Valid')) {
                table.querySelectorAll('th')[stIdx].innerHTML = 'Validation';
            }

            table.querySelectorAll('tbody tr').forEach(row => {
                if (row.dataset.universalUpgraded === 'true') return;

                let cells = row.querySelectorAll('td');
                if (cells.length < 3) return;

                let pmText = pIdx > -1 && cells[pIdx] ? cells[pIdx].innerText : "0";
                if (pmText.includes('%')) return;
                let pm = parseInt(pmText.replace(/[^0-9]/g, '')) || 0;
                let br = bIdx > -1 && cells[bIdx] ? parseInt(cells[bIdx].innerText.replace(/[^0-9]/g, '')) || 0 : 0;

                let isValid = (pm >= t_pm && br >= t_br);

                if (stIdx > -1 && cells[stIdx]) {
                    let cellText = cells[stIdx].innerText.toUpperCase();
                    if (isValid || cellText.includes('PASS')) {
                        cells[stIdx].innerHTML = `<span style="background:#2e7d32; color:white; padding:4px 12px; border-radius:12px; font-weight:bold; font-size:12px; box-shadow:0 1px 3px rgba(0,0,0,0.2);">Yes</span>`;
                        isValid = true;
                    } else {
                        cells[stIdx].innerHTML = `<span style="background:#9e9e9e; color:white; padding:4px 12px; border-radius:12px; font-weight:bold; font-size:12px; box-shadow:inset 0 1px 3px rgba(0,0,0,0.1);">No</span>`;
                        isValid = false;
                    }
                }

                function formatFulfillment(val, threshold) {
                    let pct = Math.min(100, Math.round((val / threshold) * 100));
                    let color = pct >= 100 ? '#155724' : '#721c24';
                    let bg = pct >= 100 ? '#d4edda' : '#f8d7da';
                    let border = pct >= 100 ? '#c3e6cb' : '#f5c6cb';
                    return `<div style="color:${color}; background:${bg}; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:12px; border:1px solid ${border}; display:inline-block; min-width:45px; text-align:center;">${pct}%</div>`;
                }

                if (!isValid) {
                    row.style.background = '#fdf1f1';
                    if (pIdx > -1 && cells[pIdx]) {
                        let val = parseFloat(cells[pIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                        cells[pIdx].innerHTML = formatFulfillment(val, t_pm);
                    }
                    if (bIdx > -1 && cells[bIdx]) {
                        let val = parseFloat(cells[bIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                        cells[bIdx].innerHTML = formatFulfillment(val, t_br);
                    }
                    if (drIdx > -1 && cells[drIdx]) {
                        let val = parseFloat(cells[drIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                        cells[drIdx].innerHTML = formatFulfillment(val, t_dr);
                    }
                    if (vIdx > -1 && cells[vIdx]) {
                        let val = parseFloat(cells[vIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                        cells[vIdx].innerHTML = formatFulfillment(val, t_vol);
                    }
                } else {
                    if (ancIdx > -1 && cells[ancIdx]) {
                        let originalAncestor = cells[ancIdx].innerText.trim();
                        let isGap = originalAncestor && !originalAncestor.includes('1509') && !originalAncestor.includes('Fauconer');

                        let newAncHTML = `<div style="color:#004d40; font-weight:bold; font-size:14px;">Thomas Yates (1509 - 1565) & Elizabeth Fauconer</div>`;
                        if (isGap && originalAncestor !== "Thomas Yates (1509 - 1565) & Elizabeth Fauconer") {
                            newAncHTML += `<div style="margin-top:6px; font-size:12px; color:#e65100; font-weight:bold;">[GAP: via ${originalAncestor}]</div>`;
                            newAncHTML += `<div style="margin-top:6px;"><span style="font-size:11px; color:#e65100; background:#fff3e0; padding:3px 8px; border-radius:4px; border:1px solid #ffcc80;">Network Assigned</span></div>`;
                            if (docsIdx > -1 && cells[docsIdx]) {
                                let currentDocs = cells[docsIdx].innerText.trim().replace(/\*/g, '');
                                cells[docsIdx].innerHTML = `<span style="color: #e65100; font-weight:bold;">${currentDocs}*</span>`;
                            }
                        } else {
                            newAncHTML += `<div style="margin-top:6px;"><span style="font-size:11px; color:#2e7d32; background:#e8f5e9; padding:3px 8px; border-radius:4px; border:1px solid #c8e6c9;">Network Assigned</span></div>`;
                        }
                        cells[ancIdx].innerHTML = newAncHTML;
                    }
                }
                row.dataset.universalUpgraded = 'true';
            });
        });
    }

    setTimeout(upgradePresets, 400);
    setTimeout(applyUniversalMatrixUpgrades, 600);

    const observer = new MutationObserver((mutations) => {
        let shouldRun = false;
        let shouldUpgradeTable = false;
        for(let m of mutations) {
            if(m.addedNodes.length > 0) shouldRun = true;
            if(m.target.tagName === 'TBODY' || m.target.tagName === 'TR') shouldUpgradeTable = true;
        }
        if(shouldRun) { setTimeout(upgradePresets, 100); }
        if(shouldUpgradeTable) { setTimeout(applyUniversalMatrixUpgrades, 50); }
    });
    observer.observe(document.body, { childList: true, subtree: true, characterData: true });
});
</script>
"""

# 🌟 DOM-SAFE PRECISION SPLITTER AGENT 🌟
ANCHOR_PAYLOADS['report_splitter_js'] = """
<script>
document.addEventListener('DOMContentLoaded', function() {
    let splitApplied = false;

    const reportObserver = new MutationObserver(() => {
        if (splitApplied) return;
        let cssTable = null;
        let tables = document.querySelectorAll('table');
        tables.forEach(t => {
            let ths = Array.from(t.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            if (ths.includes('PM') && ths.includes('BR') && (ths.includes('PARTICIPANT KIT') || ths.includes('PARTICIPANT'))) {
                cssTable = t;
            }
        });

        if (cssTable && !cssTable.dataset.splitted) {
            splitApplied = true;
            setTimeout(processSplits, 200);
        }
    });
    reportObserver.observe(document.body, { childList: true, subtree: true });

    function cleanupReportText() {
        // 1. SAFELY REMOVE WALL OF TEXT STRINGS
        document.querySelectorAll('p, span, b, strong, h1, h2, h3, h4, h5, li').forEach(el => {
            if(el.querySelector('table')) return; // Extreme safety guard

            let txt = el.textContent.trim();

            // Hide "Dear [Names]"
            if (txt.startsWith('Dear ') && txt.includes('[')) {
                el.style.display = 'none';
            }

            // Hide Hardcoded Summaries
            if (txt.includes('This brief evaluates the composite genetic evidence') ||
                txt.includes('The documentary paper trail is corroborated by') ||
                txt.includes('By combining these kits into a Virtual Group') ||
                txt.includes('By pooling their isolated matches into a single structural cluster')) {
                el.style.display = 'none';
            }

            // Fix Target Ancestor Language
            if (el.children.length === 0) {
                if (txt === 'Why is this the Target Ancestor?' || txt === 'Why is this the Target Node?') {
                    el.textContent = 'Why did this Ancestor Emerge?';
                }
                if (txt.includes('Inferred Target Node')) {
                    el.textContent = el.textContent.replace(/Inferred Target Node/g, 'Emerged Ancestral Node');
                }
                if (txt.includes('Target Node')) {
                    el.textContent = el.textContent.replace(/Target Node/g, 'Emerged Ancestral Node');
                }
            }
        });

        // 2. SAFELY REMOVE BULLETED LISTS
        document.querySelectorAll('p, h1, h2, h3, h4, b, strong, span').forEach(el => {
            let txt = el.textContent.trim();
            if (txt.includes('Subject Tester(s):') || txt.includes('Subject Tester:') || txt.includes('Kits Formally Merged for this Analysis')) {

                // Hide the text itself
                let parent = el.tagName === 'B' || el.tagName === 'STRONG' || el.tagName === 'SPAN' ? el.parentElement : el;
                if(parent && parent.tagName !== 'DIV') parent.style.display = 'none';
                else el.style.display = 'none';

                // Hunt down and hide the UL that follows it
                let nextEl = (parent || el).nextElementSibling;
                while(nextEl && nextEl.tagName !== 'UL' && nextEl.tagName !== 'OL' && nextEl.tagName !== 'TABLE' && nextEl.tagName !== 'DIV') {
                    nextEl = nextEl.nextElementSibling;
                }
                if(nextEl && (nextEl.tagName === 'UL' || nextEl.tagName === 'OL')) {
                    nextEl.style.display = 'none';
                }
            }
        });
    }

    function processSplits() {
        let t_pm = parseInt(localStorage.getItem('ANC_CFG_PM')) || 15;
        let t_br = parseInt(localStorage.getItem('ANC_CFG_COL')) || 2;
        let t_vol = parseFloat(localStorage.getItem('ANC_CFG_VOL')) || 40.0;
        let t_dr = 3.0;

        let validKits = new Set();
        let invalidKits = new Set();

        let totalKits = 0;
        let validCount = 0;
        let invalidCount = 0;
        let aggPM = 0;
        let aggVol = 0;

        // 🌟 PASS 1: PRECISION MATH FROM MEMORY (Solves the 0.0cM bug) 🌟
        let cssTable = null;
        let tables = document.querySelectorAll('table');
        tables.forEach(t => {
            let ths = Array.from(t.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            if (ths.includes('PM') && ths.includes('BR')) cssTable = t;
        });

        if (!cssTable) return;

        let ths = Array.from(cssTable.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
        let pIdx = ths.indexOf('PM');
        let bIdx = ths.indexOf('BR');
        let kitIdx = ths.findIndex(th => th.includes('PARTICIPANT'));
        if(kitIdx === -1) kitIdx = 0;

        let rows = cssTable.querySelectorAll('tbody tr');
        rows.forEach(r => {
            if (r.querySelectorAll('th').length > 0 && r.querySelectorAll('td').length === 0) return;
            let cells = r.querySelectorAll('td');
            if (cells.length > Math.max(pIdx, bIdx, kitIdx)) {
                let kitName = cells[kitIdx].innerText.trim();
                let pmText = cells[pIdx].innerText;

                if (pmText.includes('%')) return; // Ignore if already processed by Universal Agent

                let pm = parseInt(pmText.replace(/[^0-9]/g, '')) || 0;
                let br = parseInt(cells[bIdx].innerText.replace(/[^0-9]/g, '')) || 0;

                // Get precise volume from Window.DB
                let vol = 0;
                if(window.DB) {
                    let cleanName = kitName.split('[')[0].trim().toLowerCase();
                    let cleanCodeMatch = kitName.match(/\\[(.*?)\\]/);
                    let cleanCode = cleanCodeMatch ? cleanCodeMatch[1].trim().toLowerCase() : "";

                    let dbRows = window.DB.filter(dbRow => {
                        let rName = String(dbRow.Participant_Name || '').toLowerCase();
                        let rCode = String(dbRow.Participant_Code || dbRow.Tester_Code || '').toLowerCase();
                        return (rName === cleanName || (cleanCode && rCode === cleanCode));
                    });
                    dbRows.forEach(dbRow => {
                        vol += parseFloat(String(dbRow.cM || dbRow.Shared_cM || '0').replace(/[^0-9.]/g, ''));
                    });
                }

                totalKits++;
                if (pm >= t_pm && br >= t_br) {
                    validKits.add(kitName);
                    validCount++;
                    aggPM += pm;
                    aggVol += vol;
                } else {
                    invalidKits.add(kitName);
                    invalidCount++;
                }
            }
        });

        if (totalKits === 0) return;

        // Strip the old UI text
        cleanupReportText();

        // 🌟 PASS 2: INJECT PRECISION POPULATION SUMMARY 🌟
        let failureRate = Math.round((invalidCount / totalKits) * 100);
        let volPct = Math.round((aggVol / t_vol) * 100);

        let summaryHtml = `
            <div style="background:#f4f8f9; border-left:5px solid #006064; padding:25px; margin:30px 0; border-radius:6px; box-shadow:0 2px 8px rgba(0,0,0,0.08);">
                <h3 style="color:#004d40; margin-top:0; font-family:sans-serif; font-size:22px;">Population Sampling & Test Bed Viability</h3>

                <p style="font-size:15px; color:#444; line-height:1.6; margin-bottom:15px;">
                    This analysis evaluates the composite genetic evidence of a Virtual Group comprising <b>${totalKits}</b> individual DNA testers.
                    The methodology requires strict adherence to minimum genetic inclusion thresholds (PM &ge; ${t_pm}, BR &ge; ${t_br}) to ensure the mathematical integrity of the network.
                </p>

                <p style="font-size:15px; color:#444; line-height:1.6; margin-bottom:15px;">
                    Out of <b>${totalKits}</b> total tests evaluated, <b>${validCount}</b> met all inclusion thresholds and were formally merged into the Validated Virtual Group.
                    The remaining <b>${invalidCount}</b> kits failed to meet minimums and were cataloged as developing hypotheses.
                    This yields a current <strong>test bed failure rate of ${failureRate}%</strong>. While this population database is continually evolving, maintaining this rigorous exclusion rate prevents unverified signals from artificially inflating the lineage proof.
                </p>

                <div style="background:#e8f5e9; border:1px solid #c8e6c9; padding:15px; border-radius:6px; margin-top:15px;">
                    <p style="font-size:15px; color:#1b5e20; line-height:1.6; margin-bottom:0;">
                        The documentary paper trail is corroborated by a Validated Virtual Group of <b>${validCount} independent participant kits</b> sharing a mathematically validated aggregate of <b>${aggVol.toFixed(1)} cM</b> of Autosomal DNA. <span style="color:#2e7d32; font-weight:bold; background:#fff; padding:4px 10px; border-radius:4px; margin-left:5px; box-shadow:0 1px 3px rgba(0,0,0,0.1);">(${volPct}% FULFILLED)</span>
                    </p>
                </div>
            </div>
        `;

        let firstTable = document.querySelector('table');
        if (firstTable) {
            let wrapper = document.createElement('div');
            wrapper.innerHTML = summaryHtml;
            firstTable.parentNode.insertBefore(wrapper, firstTable);
        }

        // 🌟 PASS 3: SPLIT ALL MATRICES INTO A & B 🌟
        let isFirstTable = true;
        tables.forEach(table => {
            let tableThs = Array.from(table.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            let tKitIdx = tableThs.findIndex(th => th.includes('PARTICIPANT'));
            if (tKitIdx === -1) return;

            table.dataset.splitted = 'true';

            let stIdx = tableThs.findIndex(th => th === 'ST' || th === 'LVS' || th === 'STATUS' || th.includes('VALID'));
            if (stIdx > -1) {
                let theadThs = table.querySelectorAll('thead th');
                if (theadThs[stIdx]) theadThs[stIdx].innerText = 'Validation';
            }

            let tbody = table.querySelector('tbody') || table;
            let tRows = Array.from(tbody.querySelectorAll('tr'));

            let validRows = [];
            let invalidRows = [];

            let tPIdx = tableThs.indexOf('PM');
            let tBIdx = tableThs.indexOf('BR');
            let tVIdx = tableThs.findIndex(th => th.includes('VOL') || th.includes('CM'));
            let tDrIdx = tableThs.indexOf('DR');
            let ancIdx = tableThs.findIndex(th => th.includes('ANCESTOR') || th.includes('LINEAGE') || th.includes('NODE'));
            let docsIdx = tableThs.findIndex(th => th === 'DOCS' || th.includes('PAPER') || th.includes('SCORE') && !th.includes('CSS') && !th.includes('ANCHOR'));

            let tableHasGaps = false;

            tRows.forEach(r => {
                if (r.querySelectorAll('th').length > 0 && r.querySelectorAll('td').length === 0) return;
                let cells = r.querySelectorAll('td');
                if (cells.length > tKitIdx) {
                    let kitName = cells[tKitIdx].innerText.trim();
                    let rClone = r.cloneNode(true);
                    let cloneCells = rClone.querySelectorAll('td');

                    let isTableA = validKits.has(kitName);
                    let isGap = false;
                    let originalAncestor = "";

                    if (ancIdx > -1 && cloneCells[ancIdx]) {
                        originalAncestor = cloneCells[ancIdx].innerText.trim();
                        if (originalAncestor && !originalAncestor.includes('1509') && !originalAncestor.includes('Fauconer') && !originalAncestor.includes('Thomas Yates (1509')) {
                            isGap = true;
                        }
                    }

                    if (stIdx > -1 && cloneCells[stIdx]) {
                        if (isTableA) {
                            cloneCells[stIdx].innerHTML = `<span style="background:#2e7d32; color:white; padding:4px 12px; border-radius:12px; font-weight:bold; font-size:12px; box-shadow:0 1px 3px rgba(0,0,0,0.2);">Yes</span>`;
                        } else {
                            cloneCells[stIdx].innerHTML = `<span style="background:#9e9e9e; color:white; padding:4px 12px; border-radius:12px; font-weight:bold; font-size:12px; box-shadow:inset 0 1px 3px rgba(0,0,0,0.1);">No</span>`;
                        }
                    }

                    if (isTableA && ancIdx > -1 && cloneCells[ancIdx]) {
                        let newAncHTML = `<div style="color:#004d40; font-weight:bold; font-size:14px;">Thomas Yates (1509 - 1565) & Elizabeth Fauconer</div>`;
                        if (isGap && originalAncestor !== "Thomas Yates (1509 - 1565) & Elizabeth Fauconer") {
                            newAncHTML += `<div style="margin-top:6px; font-size:12px; color:#e65100; font-weight:bold;">[GAP: via ${originalAncestor}]</div>`;
                            newAncHTML += `<div style="margin-top:6px;"><span style="font-size:11px; color:#e65100; background:#fff3e0; padding:3px 8px; border-radius:4px; border:1px solid #ffcc80;">Network Assigned</span></div>`;
                            tableHasGaps = true;
                        } else {
                            newAncHTML += `<div style="margin-top:6px;"><span style="font-size:11px; color:#2e7d32; background:#e8f5e9; padding:3px 8px; border-radius:4px; border:1px solid #c8e6c9;">Network Assigned</span></div>`;
                        }
                        cloneCells[ancIdx].innerHTML = newAncHTML;
                    }

                    if (isTableA && isGap && docsIdx > -1 && cloneCells[docsIdx]) {
                        let currentDocs = cloneCells[docsIdx].innerText.trim().replace(/\\*/g, '');
                        cloneCells[docsIdx].innerHTML = `<span style="color: #e65100; font-weight:bold;">${currentDocs}*</span>`;
                    }

                    function formatFulfillment(val, threshold) {
                        let pct = Math.min(100, Math.round((val / threshold) * 100));
                        let color = pct >= 100 ? '#155724' : '#721c24';
                        let bg = pct >= 100 ? '#d4edda' : '#f8d7da';
                        let border = pct >= 100 ? '#c3e6cb' : '#f5c6cb';
                        return `<div style="color:${color}; background:${bg}; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:12px; border:1px solid ${border}; display:inline-block; min-width:45px; text-align:center;">${pct}%</div>`;
                    }

                    if (isTableA) {
                        validRows.push(rClone);
                    } else {
                        if (tPIdx > -1 && cloneCells[tPIdx]) {
                            let val = parseFloat(cloneCells[tPIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                            cloneCells[tPIdx].innerHTML = formatFulfillment(val, t_pm);
                        }
                        if (tBIdx > -1 && cloneCells[tBIdx]) {
                            let val = parseFloat(cloneCells[tBIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                            cloneCells[tBIdx].innerHTML = formatFulfillment(val, t_br);
                        }
                        if (tDrIdx > -1 && cloneCells[tDrIdx]) {
                            let val = parseFloat(cloneCells[tDrIdx].innerText.replace(/[^0-9.]/g, '')) || 0;
                            cloneCells[tDrIdx].innerHTML = formatFulfillment(val, t_dr);
                        }
                        invalidRows.push(rClone);
                    }
                }
            });

            buildSplitUI(table, validRows, invalidRows,
                "Validated Lineages Matrix (Virtual Group)",
                "These kits met all minimum inclusion thresholds and were formally merged into the Virtual Group to build the aggregate validation case.",
                "Developing & Hypothesis Catalog",
                "These kits failed to meet the minimum Proper Match (PM) or Branching (BR) inclusion rules prior to analysis. Per methodological standards, they are cataloged as Developing, but their DNA was not used to build the aggregated network evidence.",
                isFirstTable, tableHasGaps, t_pm, t_br, t_dr, tableThs.length);
            isFirstTable = false;
        });
    }

    function buildSplitUI(originalTable, validRows, invalidRows, titleA, descA, titleB, descB, isFirst, hasGaps, t_pm, t_br, t_dr, colSpan) {
        let wrapper = document.createElement('div');
        let theadHTML = originalTable.querySelector('thead') ? originalTable.querySelector('thead').outerHTML : '';

        let explanationHtml = '';
        if (isFirst) {
            explanationHtml = `
            <div style="background:#fff; border-radius:8px; box-shadow:0 4px 15px rgba(0,0,0,0.05); padding:30px; margin-bottom:40px; border-left:6px solid #004d40;">
                <h3 style="color:#004d40; margin-top:0; font-family:sans-serif; font-size:22px;">Matrix Architecture Explanation</h3>
                <p style="font-size:15px; line-height:1.7; color:#333;">
                To preserve methodological integrity, the Yates DNA Study separates participants into two analytical groups.
                Only participants who meet the minimum genetic inclusion thresholds are allowed to contribute to the
                aggregated network evidence used to evaluate lineage convergence.
                </p>
                <p style="font-size:15px; line-height:1.7; color:#333;">
                Participants who do not yet meet these thresholds are still recorded and analyzed, but their DNA results
                are cataloged as developing hypotheses rather than contributing to the formal validation calculations.
                This separation ensures that incomplete or weak signals do not distort the measurement of the underlying
                genetic network.
                </p>
                <h4 style="margin-top:24px; color:#222; font-size:18px;">Table A — Validated Lineages Matrix (Virtual Group)</h4>
                <p style="font-size:15px; line-height:1.7; color:#333;">
                Participants appearing in this table satisfied the minimum inclusion criteria and were formally merged into
                the <b>Virtual Group</b>. Their combined results form the aggregated network used to evaluate the emergence
                of the consensus ancestral node.
                </p>
                <ul style="margin:10px 0 0 18px; line-height:1.7; color:#333; font-size:15px;">
                <li><b>Proper Matches (PM):</b> ${t_pm} or greater (Ensures minimum evidence mass)</li>
                <li><b>Independent Branches (BR):</b> ${t_br} or greater (Prevents isolated family-line bias)</li>
                <li><b>Dominance Ratio (DR):</b> ${t_dr} or greater (Indicates clear lineage preference)</li>
                </ul>
                <p style="font-size:15px; line-height:1.7; color:#333; margin-top:10px;">
                Because these participants meet the required thresholds, their genetic scores support placement within the
                emerged lineage network.
                </p>
                <h4 style="margin-top:24px; color:#222; font-size:18px;">Table B — Developing & Hypothesis Catalog</h4>
                <p style="font-size:15px; line-height:1.7; color:#333;">
                Participants listed in this table did not yet meet one or more minimum inclusion thresholds at the time
                of analysis. Their results are retained as research leads, but their DNA was not used to construct the
                aggregated network evidence for the lineage validation case.
                </p>
                <p style="font-size:15px; line-height:1.7; color:#333;">
                Instead of raw metric values, this table displays <b>percentage fulfillment</b> for key validation metrics.
                This allows readers to see how close each participant is to meeting the inclusion thresholds.
                </p>
                <div style="background:#f7fbfa; border-left:4px solid #00acc1; padding:14px 16px; margin-top:20px; margin-bottom:16px;">
                <b style="color:#006064;">Interpretation:</b><br>
                <span style="font-size:14px; line-height:1.6; color:#444;">The separation between validated and developing participants protects the integrity of the analysis.
                It ensures that the aggregated lineage evidence is built only from participants whose DNA networks meet
                minimum replication and branching standards.</span>
                </div>
                <p style="font-size:15px; line-height:1.7; color:#333; margin-bottom:0; font-style:italic;">
                This approach mirrors the way population genetics studies separate validated sample cohorts from exploratory observations.
                </p>
            </div>
            `;
        }

        let footnoteHtml = '';
        if (hasGaps && validRows.length > 0) {
            footnoteHtml = `
            <div style="font-size: 13px; color: #555; margin-top: 15px; padding: 12px 15px; background: #fff8e1; border-radius: 4px; border-left: 4px solid #ff9800; font-family: sans-serif; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <b style="color: #e65100;">* Pedigree Gap Flag:</b> Asterisk indicates the participant's documented spine currently fails to reach the Emerged Ancestor (Apex Reach gap). The algorithm has superseded the documentary gap based on verified network saturation.
            </div>
            `;
        }

        // Safe Fallback if 0 kits pass
        let vBodyHtml = validRows.length > 0 ? validRows.map(r => r.outerHTML).join('') : `<tr><td colspan="${colSpan || 15}" style="text-align:center; padding:20px; font-style:italic; color:#666;">No participant kits met the required inclusion thresholds.</td></tr>`;

        let html = explanationHtml + `
            <div style="margin-bottom: 40px; background: #fff; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); padding: 20px; border-top: 5px solid #2e7d32;">
                <h3 style="color: #2e7d32; margin-top: 0; font-size: 20px; display: flex; justify-content: space-between; align-items: center;">
                    Table A: ${titleA}
                    <span style="font-size: 12px; background: #e8f5e9; padding: 4px 10px; border-radius: 12px;">${validRows.length} Kits Passed</span>
                </h3>
                <p style="font-size: 14px; color: #444; background: #f4f8f9; padding: 12px; border-left: 4px solid #2e7d32; border-radius: 4px; font-weight: 500; line-height: 1.5;">
                    ${descA}
                </p>
                <div style="overflow-x:auto;">
                    <table class="${originalTable.className}" style="width:100%; border-collapse:collapse; margin-top:15px; font-size:14px;">
                        ${theadHTML.replace(/<thead/g, '<thead style="background:#2e7d32; color:white; text-align:center;"')}
                        <tbody>
                            ${vBodyHtml}
                        </tbody>
                    </table>
                </div>
                ${footnoteHtml}
            </div>
        `;

        if (invalidRows.length > 0) {
            html += `
            <div style="margin-bottom: 40px; background: #fff; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); padding: 20px; border-top: 5px solid #c62828; opacity: 0.9;">
                <h3 style="color: #c62828; margin-top: 0; font-size: 20px; display: flex; justify-content: space-between; align-items: center;">
                    Table B: ${titleB}
                    <span style="font-size: 12px; background: #ffebee; padding: 4px 10px; border-radius: 12px;">${invalidRows.length} Kits Not Validated</span>
                </h3>
                <p style="font-size: 14px; color: #444; background: #fdf1f1; padding: 12px; border-left: 4px solid #c62828; border-radius: 4px; font-weight: 500; line-height: 1.5;">
                    ${descB}
                </p>
                <div style="overflow-x:auto;">
                    <table class="${originalTable.className}" style="width:100%; border-collapse:collapse; margin-top:15px; font-size:14px;">
                        ${theadHTML.replace(/<thead/g, '<thead style="background:#c62828; color:white; text-align:center;"')}
                        <tbody>
                            ${invalidRows.map(r => r.outerHTML).join('')}
                        </tbody>
                    </table>
                </div>
            </div>
            `;
        }

        wrapper.innerHTML = html;
        originalTable.parentNode.replaceChild(wrapper, originalTable);
    }
});
</script>
"""

ANCHOR_PAYLOADS['explanation_html'] = """
<div style="max-width: 1200px; margin: 40px auto; padding: 40px; background: #ffffff; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); font-family: sans-serif; line-height: 1.7; color: #444;">
    <h2 style="color: #004d40; border-bottom: 2px solid #004d40; padding-bottom: 10px; margin-top: 0;">How the ANCHOR Framework Works</h2>
    <p style="font-size: 16px;">The <strong>ANCHOR Framework</strong> evaluates lineage hypotheses using two independent evidence systems: <strong>Genetic Evidence</strong> and <strong>Documentary Evidence</strong>. These evidence streams are measured separately and then combined to produce an overall lineage reliability assessment.</p>

    <h3 style="color: #006064; margin-top: 30px; font-size: 20px;">Genetic Evidence: Collateral Saturation (CSS)</h3>
    <p>Genetic strength is measured using Collateral Saturation metrics (CSS) derived from autosomal DNA match networks.</p>
    <p>Autosomal DNA is inherited from all ancestors and redistributed through recombination in each generation. As a result, descendants of a historical ancestor inherit different fragments of that ancestor’s genome. These fragments appear across many individuals in the autosomal match network as replicated Identity-by-Descent (IBD) signals.</p>
    <p>Collateral Saturation measures the strength of these networks through:</p>
    <ul style="background: #f4f8f9; padding: 20px 40px; border-radius: 6px; border-left: 4px solid #00acc1;">
        <li><strong>Proper Matches (PM)</strong></li>
        <li><strong>Target Handshakes (HC-T)</strong></li>
        <li><strong>Dominance Ratio (DR)</strong></li>
        <li><strong>Independent Branches (BR)</strong></li>
        <li><strong>Unique Testers (TB)</strong></li>
        <li><strong>Node Saturation (NS)</strong></li>
        <li><strong>Stability Testing</strong></li>
    </ul>
    <p>When multiple independent descendant lines carry overlapping genetic signals, the network forms a replicated genetic signature of the ancestral node. Lineage validation therefore emerges from network replication, not from any single DNA match.</p>

    <h3 style="color: #006064; margin-top: 30px; font-size: 20px;">Documentary Evidence: Pedigree Strength (DOCS)</h3>
    <p>Documentary strength is evaluated using structured pedigree analysis derived from GEDCOM data. The framework measures documentary reliability using standardized pedigree completeness metrics including:</p>
    <ul style="background: #f4f8f9; padding: 20px 40px; border-radius: 6px; border-left: 4px solid #00acc1;">
        <li>Generational depth</li>
        <li>Vital record coverage</li>
        <li>Source citation presence</li>
        <li>Geographic consistency across generations</li>
    </ul>
    <p>These factors are combined into a Documentary Strength Score (DOCS) representing the reliability of the documentary lineage.</p>

    <h3 style="color: #006064; margin-top: 30px; font-size: 20px;">Convergence of Evidence</h3>
    <p>The ANCHOR Framework operates on the principle that the strongest genealogical conclusions occur when independent evidence systems converge on the same ancestral hypothesis.</p>
    <p>Autosomal DNA networks provide biological evidence of shared ancestry, while documentary records provide historical context linking descendants to specific ancestors. When both evidence streams independently support the same lineage, the result is ANCHOR-grade lineage validation.</p>

    <h3 style="color: #006064; margin-top: 30px; font-size: 20px;">Network-Level Lineage Reconstruction</h3>
    <p>A key discovery of the Yates DNA Study is that lineage proof is a network-level phenomenon. Individual DNA kits may fail proof-grade criteria when evaluated independently. However, when multiple testers representing independent descendant branches are evaluated together, their combined genetic evidence forms a stable network.</p>
    <p>The framework therefore evaluates clusters of related testers as Virtual Groups, representing the collective genetic signal of the lineage. Through this approach, the autosomal match network becomes a distributed reconstruction of surviving ancestral genetic fragments across descendant lines.</p>

    <h3 style="color: #006064; margin-top: 30px; font-size: 20px;">Bi-Parental Lineage Discovery</h3>
    <p>Because autosomal DNA is inherited from both parents, validation of a Yates lineage simultaneously validates the maternal partner lineages connected to that ancestor. Analysis of the Yates dataset demonstrates that strong genetic signals often appear for non-Yates surnames linked to the same ancestral nodes.</p>
    <p>This reveals an important property of autosomal analysis: surname studies contain the potential to reconstruct multiple interconnected family lineages simultaneously.</p>

    <h3 style="color: #006064; margin-top: 30px; font-size: 20px;">The Purpose of ANCHOR</h3>
    <p>The ANCHOR Framework provides a replicable and scalable method for autosomal lineage reconstruction. Rather than relying on subjective interpretation of individual matches, the framework quantifies lineage evidence using measurable genetic network properties and structured documentary evaluation.</p>
    <p>By combining these independent evidence systems, ANCHOR allows researchers to move from isolated DNA matches toward structured, proof-grade lineage reconstruction.</p>

    <div style="margin-top: 40px; padding: 25px; background: #e0f2f1; border-left: 6px solid #00695c; border-radius: 6px;">
        <h4 style="margin-top: 0; color: #004d40; font-size: 18px;">ANCHOR in One Sentence</h4>
        <p style="margin-bottom: 0; font-size: 16px; color: #004d40;">The ANCHOR Framework is a robust, dual-verification system that mathematically measures the intersection of autosomal DNA network saturation and documentary pedigree strength to definitively validate ancestral lineages.</p>
    </div>
</div>
"""

ANCHOR_PAYLOADS['about_content'] = """
<div style="max-width: 1000px; margin: 40px auto; padding: 40px; background: #ffffff; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); font-family: sans-serif; line-height: 1.7; color: #444;">
    <h2 style="color: #004d40; border-bottom: 2px solid #004d40; padding-bottom: 10px; margin-top: 0;">Origin of the Methodology</h2>
    <p>The development of the ANCHOR Framework did not begin as a formal academic project. It emerged from the practical challenges encountered during the Yates DNA Study, a long-term effort to reconstruct historical Yates family lineages using autosomal DNA evidence combined with traditional genealogical research.</p>
    <p>The study initially followed the typical approach used in many genetic genealogy projects: evaluating individual DNA matches and attempting to connect them to specific ancestors through documentary research. Over time, it became clear that this approach often produced uncertain conclusions. Individual autosomal matches frequently reflect multiple ancestral pathways, and documentary pedigrees may contain incomplete information, transcription errors, or assumptions that cannot be easily verified.</p>
    <p>Through sustained analysis of the expanding Yates autosomal dataset, a different pattern began to emerge. Rather than focusing on individual matches, the analysis shifted toward examining <strong>replicated networks of matches across multiple descendant lines</strong>.</p>
    <p>When descendants of the same historical ancestor appeared within the dataset, they produced consistent patterns of shared matches that repeated across independent pedigree branches. This observation suggested that lineage validation could be approached as a <strong>network replication problem</strong> rather than a match-by-match interpretation problem.</p>
    <p>The methodological perspective that ultimately produced the ANCHOR Framework reflects the author’s background in analytical systems thinking and structured problem-solving. Instead of treating DNA matches as isolated data points, the Yates DNA Study began treating the autosomal match dataset as a <strong>networked system</strong> in which patterns of replication, density, and stability could be measured.</p>
    <p>This shift led to the development of the <strong>Collateral Saturation method</strong>, which evaluates the strength of lineage hypotheses by measuring the density and replication of descendant match networks across independent branches of a family line.</p>
    <p>As the study progressed, it also became clear that genetic evidence and documentary genealogy needed to be evaluated together. Strong genetic networks required documentary context to identify specific ancestors, while well-documented pedigrees required genetic confirmation to establish biological validity.</p>
    <p>The <strong>ANCHOR Framework (Autosomal Network Corroboration for Historical Origin Reconstruction)</strong> emerged as the integration of these two evidence systems. Within the framework:</p>
    <ul style="background: #f4f8f9; padding: 20px 40px; border-radius: 6px; border-left: 4px solid #00acc1;">
        <li>Collateral Saturation metrics measure the strength and stability of autosomal DNA match networks.</li>
        <li>Structured GEDCOM analysis measures the completeness and reliability of documentary pedigrees.</li>
    </ul>
    <p>Together these components provide a reproducible methodology for evaluating lineage hypotheses through the convergence of independent genetic and documentary evidence. The Yates DNA Study represents the first large-scale implementation of this framework and demonstrates that autosomal DNA match networks, when analyzed systematically, can support rigorous lineage reconstruction across multiple descendant branches.</p>

    <div style="margin: 40px auto; max-width: 800px; text-align: center; background: #eceff1; padding: 20px; border-radius: 8px; border: 2px solid #b0bec5;">
        <h3 style="color: #004d40; margin-top: 0; margin-bottom: 15px; font-size: 18px;">🎥 ANCHOR Framework Video Presentation</h3>
        <video width="100%" controls style="border-radius: 6px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
            <source src="https://yates.one-name.net/anchor/anchor-hub/media/all/A_500-Year-Old_Mystery.mp4" type="video/mp4">
            Your browser does not support the video tag.
        </video>
    </div>

    <h2 style="color: #004d40; border-bottom: 2px solid #004d40; padding-bottom: 10px; margin-top: 40px;">About the Researcher</h2>
    <p>Ronald Eugene Yates developed the ANCHOR Framework during the course of the Yates DNA Study, a One-Name Study focused on reconstructing historical Yates lineages through autosomal DNA analysis and documentary genealogy.</p>
    <p>The research approach reflects a professional background grounded in analytical problem solving, systems thinking, and structured evaluation of complex datasets. These perspectives influenced the development of a methodology that treats autosomal DNA match lists not as isolated observations but as components of a larger interconnected network.</p>
    <p>The Yates DNA Study applies these principles to genealogical research by combining large autosomal match datasets with structured pedigree information derived from GEDCOM records. Through iterative analysis and repeated testing of lineage hypotheses, the study developed the Collateral Saturation method and the broader ANCHOR Framework.</p>
    <p>This work represents an effort to move autosomal DNA research toward <strong>replicable analytical methods</strong>, allowing lineage conclusions to be evaluated through measurable network properties rather than subjective interpretation of individual matches.</p>
    <p>The ANCHOR Framework continues to evolve as the Yates DNA dataset expands and additional descendant lines are incorporated into the study.</p>

    <div style="margin: 40px; padding: 25px; background: #e0f2f1; border-left: 6px solid #00695c; border-radius: 6px; text-align: center;">
        <p style="margin-bottom: 0; font-size: 16px; color: #004d40; font-style: italic;">"The ANCHOR Framework represents a transition in autosomal genealogy from isolated match interpretation toward <strong>structured network-based lineage reconstruction</strong>, where historical ancestry is validated through the replicated genetic signals of multiple independent descendants."</p>
    </div>
</div>
"""

print("✅ [CELL 5A] Bulletproof Text Scrubber loaded safely into memory! Please run Cell 5B to publish.")

      [CELL 5A] LOADING UI PAYLOADS INTO MEMORY...
✅ [CELL 5A] Bulletproof Text Scrubber loaded safely into memory! Please run Cell 5B to publish.


<>:210: SyntaxWarning: invalid escape sequence '\*'
<>:210: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipykernel_184/1813609292.py:210: SyntaxWarning: invalid escape sequence '\*'
  let currentDocs = cells[docsIdx].innerText.trim().replace(/\*/g, '');


In [27]:
# @title [CELL 5B] The Publisher Engine (Universal Injection)
def run_publisher():
    print("="*60)
    print("      [CELL 5B] PUBLISHER STARTING (Jinja2 Engine Active)...")
    print("="*60)

    import os, json, pytz, urllib.request, urllib.parse, re
    import pandas as pd
    from jinja2 import Template
    from datetime import datetime
    from ftplib import FTP_TLS
    try:
        from google.colab import userdata
    except ImportError:
        pass

    if 'SITE_TEMPLATES' not in globals() or 'ANCHOR_PAYLOADS' not in globals():
        print("\n❌ STOPPING PUBLISHER: Memory Wipe Detected! ❌")
        return print("    👉 FIX: Please run Cell 3, Cell 4, and Cell 5A to load your templates.")

    try:
        HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
        USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
        PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
    except Exception as e:
        return print(f"❌ Credential Error: {e}")

    JSON_DB = "compiled_database.json"
    CSV_DB = "engine_database.csv"
    est = pytz.timezone('US/Eastern')

    if not os.path.exists(JSON_DB):
        return print("❌ ERROR: compiled_database.json not found. Run Cell 2 first.")

    with open(JSON_DB, "r") as f:
        master_data = json.load(f)

    pre_str = json.dumps(master_data.get('PRECOMPUTED', []))
    data_str = json.dumps(master_data.get('DATA', {}))
    db_str = json.dumps(master_data.get('DB', []))
    cache_buster = int(datetime.now().timestamp())
    js_file_content = f"window.PRECOMPUTED={pre_str};\nwindow.DATA={data_str};\nwindow.DB={db_str};\n"
    js_file_content = re.sub(r'\bNone\s*\(I(\d+)\)', r'Unnamed Ancestor (I\1)', js_file_content)
    db_script_tag = f'<script src="database.js?v={cache_buster}"></script>'

    try:
        df = pd.read_csv(CSV_DB, encoding="iso-8859-15")
    except pd.errors.EmptyDataError:
        return print(f"❌ ERROR: {CSV_DB} is empty.")

    df = df.fillna('')
    df = df.replace('nan', '')

    if "Authority_Directory_Label" in df.columns: df.rename(columns={"Authority_Directory_Label": "Dir_Label"}, inplace=True)
    for col in ["Tester_Display", "Kit_Name", "Participant_Name"]:
        if col in df.columns:
            df.rename(columns={col: "Participant_Name"}, inplace=True)
            df['Participant_Name'] = df['Participant_Name'].apply(lambda x: str(x).split('[')[0].split('(')[0].strip())
            break
    for col in ["Tester_Code", "Kit_Code", "Participant_Code"]:
        if col in df.columns:
            df.rename(columns={col: "Participant_Code"}, inplace=True)
            df['Participant_Code'] = df['Participant_Code'].apply(lambda x: str(x).strip())
            break
    for col in ["Match_Lineage", "Lineage"]:
        if col in df.columns:
            df.rename(columns={col: "Lineage"}, inplace=True)
            break

    df_valid = df[df['Dir_Label'] != 'No Matches'].copy()
    true_counts = {}
    for _, row in df_valid.iterrows():
        p_name = str(row.get('Participant_Name', '')).strip().lower()
        p_code = str(row.get('Participant_Code', '')).strip().lower()
        if p_name: true_counts[p_name] = true_counts.get(p_name, 0) + 1
        if p_code: true_counts[p_code] = true_counts.get(p_code, 0) + 1
    mc_dir = df_valid['Dir_Label'].value_counts()

    print("    [*] Connecting to FTP Server for Data Sweeps...")
    media_links = []
    try:
        fetch_ftp = FTP_TLS()
        fetch_ftp.connect(HOST, 21); fetch_ftp.auth(); fetch_ftp.login(USER, PASS); fetch_ftp.prot_p()

        try:
            fetch_ftp.cwd("/anchor/anchor-hub/media/all")
            media_files = fetch_ftp.nlst()
            for f in sorted(media_files):
                if f.lower().endswith(('.mp4', '.m4a', '.pdf', '.pptx', '.png', '.jpg', '.jpeg')) and not f.startswith('.'):
                    clean_name = f.rsplit('.', 1)[0].replace('_', ' ')
                    ext = f.rsplit('.', 1)[1].upper()
                    encoded_f = urllib.parse.quote(f)
                    url = f"https://yates.one-name.net/anchor/anchor-hub/media/all/{encoded_f}"
                    icon = "📄"
                    if ext in ['MP4']: icon = "🎥"
                    elif ext in ['M4A']: icon = "🎧"
                    elif ext in ['PNG', 'JPG', 'JPEG']: icon = "🖼️"
                    elif ext in ['PPTX']: icon = "📊"
                    media_links.append(f'<a href="{url}" target="_blank" style="display:inline-flex; align-items:center; background:rgba(255,255,255,0.08); border-radius:20px; padding:6px 14px; color:#e0f7fa; text-decoration:none; font-weight:600; font-size:13px; border:1px solid rgba(255,255,255,0.15); box-shadow:0 2px 4px rgba(0,0,0,0.1); transition:all 0.2s;" onmouseover="this.style.background=\'rgba(255,255,255,0.15)\'; this.style.borderColor=\'#80cbc4\'; this.style.transform=\'translateY(-1px)\'" onmouseout="this.style.background=\'rgba(255,255,255,0.08)\'; this.style.borderColor=\'rgba(255,255,255,0.15)\'; this.style.transform=\'none\'"><span style="margin-right:6px; font-size:14px;">{icon}</span> {clean_name} <span style="font-size:10px; color:#80cbc4; margin-left:6px; background:rgba(0,0,0,0.2); padding:2px 6px; border-radius:10px;">{ext}</span></a>')
        except: pass

        try: fetch_ftp.cwd("/")
        except: pass
        try: fetch_ftp.cwd("/anchor/anc-1000-yates")
        except: pass

        fetched_auth = False
        for auth_name in ["masked_to_unmasked.csv", "match_to_unmasked.csv", "match_to_unmaskedEXP.csv"]:
            try:
                with open(auth_name, "wb") as f: fetch_ftp.retrbinary(f"RETR {auth_name}", f.write)
                fetched_auth = True
                break
            except: pass
        fetch_ftp.quit()
    except: pass

    auth_df = pd.DataFrame()
    for fname in ["masked_to_unmasked.csv", "match_to_unmasked.csv", "match_to_unmaskedEXP.csv"]:
        if os.path.exists(fname) and os.path.getsize(fname) > 0:
            try: auth_df = pd.read_csv(fname); auth_df.columns = auth_df.columns.str.strip(); break
            except: continue

    admin_rows = []
    zero_rows = []
    if not auth_df.empty:
        for _, row in auth_df.iterrows():
            p_code = str(row.get('Sort_Key_A', row.get('archive', row.get('code', '')))).strip()
            p_name = str(row.get('Display_Name', row.get('unmasked', row.get('Unmasked', '')))).strip()
            p_id = str(row.get('Sort_Key_B', row.get('#id', row.get('ID', '')))).strip()
            sort_key = str(row.get('Sort_Key_C', row.get('Authority_Sort_Key', p_name))).strip().lower()
            num_assign = str(row.get('Numeric_assigned', '')).strip()
            person_key = f"{num_assign.replace('-', '')}{sort_key[:7]}" if num_assign.replace('-', '') else ""
            count = true_counts.get(p_name.lower(), 0)
            if count == 0 and p_code: count = true_counts.get(p_code.lower(), 0)
            item = {'name': p_name, 'code': p_code, 'id': p_id, 'count': count, 'sort': sort_key, 'person_key': person_key}
            admin_rows.append(item)
            if count == 0: zero_rows.append(item)
    else:
        for p_name, p_info in master_data['DATA'].get('participants', {}).items():
            pure_name = str(p_name).split('[')[0].strip()
            pure_code = str(p_info.get('kit_code', '')).strip()
            count = true_counts.get(pure_name.lower(), 0)
            if count == 0 and pure_code: count = true_counts.get(pure_code.lower(), 0)
            item = {'name': pure_name, 'code': pure_code, 'id': '', 'count': count, 'sort': str(p_info.get('sort_key', pure_name)).lower(), 'person_key': ''}
            admin_rows.append(item)
            if count == 0: zero_rows.append(item)

    admin_rows.sort(key=lambda x: x['sort'])
    zero_rows.sort(key=lambda x: x['sort'])
    num_participants = len(admin_rows)

    def get_auth_data(r_code, r_name):
        c_low = str(r_code).strip().lower()
        n_low = str(r_name).strip().lower()
        for row in admin_rows:
            if row['code'].lower() == c_low and c_low != '': return row['name'], row['sort'], row['person_key']
            if row['name'].lower() == n_low and n_low != '': return row['name'], row['sort'], row['person_key']
        return r_name, n_low, ""

    auth_results = df_valid.apply(lambda x: get_auth_data(x.get('Participant_Code', ''), x.get('Participant_Name', '')), axis=1)
    df_valid['Participant_Name'] = [res[0] for res in auth_results]
    df_valid['sort_key'] = [res[1] for res in auth_results]
    df_valid['Person_Key'] = [res[2] for res in auth_results]

    unique_matches = sorted(df_valid['Match_Name'].dropna().unique(), key=lambda x: str(x).lower())
    match_token_map, match_alpha_map, m_counter = {}, {}, 5
    for m_name in unique_matches:
        clean_name = re.sub(r'[^a-zA-Z0-9 ]', '', str(m_name)).strip().lower()
        parts = clean_name.split()
        alpha_sort = parts[-1] + "".join(parts[:-1]) if len(parts) > 1 else clean_name
        match_token_map[m_name] = f"ancm{m_counter:03d}{alpha_sort[:7]}"
        match_alpha_map[m_name] = alpha_sort
        m_counter += 5

    df_valid['Match_Token'] = df_valid['Match_Name'].map(match_token_map)
    df_valid['Match_Alpha'] = df_valid['Match_Name'].map(match_alpha_map)
    df_valid['Dir_Label_Alpha'] = df_valid['Dir_Label'].apply(lambda x: re.sub(r'[^a-z0-9]', '', str(x).split('(')[0].lower()))

    def clean_cm(val):
        try: return float(re.sub(r'[^\d.]', '', str(val)))
        except: return 0.0
    df_valid['numeric_cM'] = df_valid['cM'].apply(clean_cm)

    matrix_data = []
    for label, group in df_valid.groupby('Dir_Label'):
        if label == 'No Matches' or str(label).strip() == '': continue
        matrix_data.append({'Emerged Ancestor': f"<strong style='color:#006064; font-size:15px;'>{str(label).split('(')[0].strip()}</strong>", 'Genetic Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(CSS v2a)</span>': '', 'Documentary Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(DOCS)</span>': '', 'ANCHOR Score': '', '_sort_key': str(label).lower()})
    df_matrix = pd.DataFrame(matrix_data).sort_values(by='_sort_key').drop(columns=['_sort_key'])
    matrix_html_table = df_matrix.to_html(index=False, escape=False, classes="matrix-table", border=0)

    def normalize_id(val): return f"I{str(val).replace('@', '').strip()}" if str(val).replace('@', '').strip().isdigit() else str(val).replace('@', '').strip()

    def format_reg(r):
        m_id, m_name, d_label, cm_val = str(r.get("Match_ID", "")), str(r.get("Match_Name", "")), str(r.get("Dir_Label", "")).split('(')[0].strip(), str(r.get("cM", ""))
        p_disp = str(r.get("Person_Key", "")) or str(r.get("Participant_Name", ""))
        m_disp = str(r.get("Match_Token", m_name))
        lin_len = len(str(r.get("Lineage", "")).split("->"))
        text = f"<b>{p_disp}</b> is a {cm_val} cM match to <a href='https://yates.one-name.net/tng/verticalchart.php?personID={normalize_id(m_id)}&tree=tree1&parentset=0&display=vertical&generations=15' target='_blank'><b>{m_disp}</b></a> via {d_label} back {lin_len} generations."
        if mc_dir.get(r['Dir_Label'], 0) == 1: return f"<div style='display:flex; align-items:center; justify-content:space-between; width:100%;'><div style='padding-right:15px; line-height:1.5;'>{text}</div><div style='flex-shrink:0; font-size:0.85em; color:#e65100; font-weight:bold; background:#fff8e1; padding:4px 10px; border-radius:4px; border:1px solid #ffe082; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>🌟 Singleton Line</div></div>"
        return f"<div style='line-height:1.5;'>{text}</div>"

    def format_tree(r):
        m_id, m_name_raw, lin_str = str(r.get("Match_ID", "")), str(r.get("Match_Name", "")), str(r.get("Lineage", ""))
        p_disp = str(r.get("Person_Key", "")) or str(r.get("Participant_Name", ""))
        m_disp = str(r.get("Match_Token", m_name_raw))
        linked_lin = lin_str.replace(m_name_raw, f'<a href="https://yates.one-name.net/tng/verticalchart.php?personID={normalize_id(m_id)}&tree=tree1&parentset=0&display=vertical&generations=15" target="_blank" style="color:#006064;text-decoration:none;font-weight:bold;">{m_disp}</a>') if m_name_raw in lin_str else lin_str
        text = f"<b style='color:#4a148c;'>{p_disp}</b>: {linked_lin}"
        if mc_dir.get(r['Dir_Label'], 0) == 1: return f"<div style='display:flex; align-items:center; justify-content:space-between; width:100%;'><div style='padding-right:15px; line-height:1.5;'>{text}</div><div style='flex-shrink:0; font-size:0.85em; color:#e65100; font-weight:bold; background:#fff8e1; padding:4px 10px; border-radius:4px; border:1px solid #ffe082; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>🌟 Singleton Line</div></div>"
        return f"<div style='line-height:1.5;'>{text}</div>"

    df_valid['Reg_Narrative'] = df_valid.apply(format_reg, axis=1)
    df_valid['Tree_Narrative'] = df_valid.apply(format_tree, axis=1)

    df_reg_za = df_valid.sort_values(by=['Dir_Label_Alpha', 'sort_key'], ascending=[False, True]).copy()
    df_reg_za.rename(columns={'Reg_Narrative': 'Participant Matches & Lineages'}, inplace=True)
    df_reg_az = df_valid.sort_values(by=['sort_key', 'Dir_Label_Alpha'], ascending=[True, False]).copy()
    df_reg_az.rename(columns={'Reg_Narrative': 'Participant Matches & Lineages'}, inplace=True)
    df_reg_match = df_valid.sort_values(by=['Match_Alpha', 'sort_key'], ascending=[True, True]).copy()
    df_reg_match.rename(columns={'Reg_Narrative': 'Participant Matches & Lineages'}, inplace=True)

    df_tree_za = df_valid.sort_values(by=['Dir_Label_Alpha', 'sort_key'], ascending=[False, True]).copy()
    df_tree_za.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)
    df_tree_az = df_valid.sort_values(by=['sort_key', 'Dir_Label_Alpha'], ascending=[True, False]).copy()
    df_tree_az.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)
    df_tree_match = df_valid.sort_values(by=['Match_Alpha', 'sort_key'], ascending=[True, True]).copy()
    df_tree_match.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)

    print("    [+] Fetching Central Master Template from London Hub...")
    pages_to_upload = {}
    req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}

    try:
        req_master = urllib.request.Request("https://yates.one-name.net/anchor/anchor-hub/template_master.html", headers=req_headers)
        master_jinja = Template(urllib.request.urlopen(req_master).read().decode('utf-8'))
    except Exception as e:
        return print(f"❌ JINJA ERROR: Failed to download Master Template. Error: {e}")

    nav_html = SITE_TEMPLATES.get('nav', '')

    if 'about.shtml' not in nav_html:
        match = re.search(r'(<a[^>]*href="[^"]*(?:contents|research_admin)\.[^"]*"[^>]*>.*?</a>)', nav_html)
        if match:
            clone_link = match.group(1)
            about_link = re.sub(r'href="[^"]*"', 'href="about.shtml"', clone_link)
            about_link = re.sub(r'>.*?</a>', '>About</a>', about_link)
            nav_html = nav_html.replace(clone_link, f"{about_link}\n{clone_link}")
        else:
            nav_html += ' | <a href="about.shtml">About</a>'

    nav_html = re.sub(r'<a[^>]*href="[^"]*research_admin[^"]*"[^>]*>\s*Admin Hub\s*</a>\s*(?:\||&nbsp;)*', '', nav_html, flags=re.IGNORECASE)
    nav_html = re.sub(r'<a[^>]*href="[^"]*admin[^"]*"[^>]*>\s*Admin\s*</a>', r'<a href="research_admin.html" style="color:#ffcc80; font-weight:bold; text-decoration:none;" title="Anchor Engine Admin">Admin</a>', nav_html, flags=re.IGNORECASE)
    if 'research_admin.html' not in nav_html:
        nav_html = re.sub(r'<a[^>]*>\s*Admin\s*</a>', r'<a href="research_admin.html" style="color:#ffcc80; font-weight:bold; text-decoration:none;" title="Anchor Engine Admin">Admin</a>', nav_html, flags=re.IGNORECASE)

    media_sub_header = ""
    if media_links:
        media_sub_header = f"""
        <div class="no-print" style="background:#004d40; border-top:1px solid rgba(255,255,255,0.1); padding:12px 15px; border-bottom: 2px solid #e65100;">
            <div style="display:flex; flex-wrap:wrap; align-items:center; justify-content:center; gap:12px; max-width:1400px; margin:0 auto; font-family:sans-serif;">
                <span style="color:#ffcc80; font-size:13px; font-weight:bold; text-transform:uppercase; letter-spacing:1px; margin-right:5px; text-shadow:0 1px 2px rgba(0,0,0,0.2);">🎙️ AI Media Hub</span>
                {''.join(media_links)}
            </div>
        </div>
        """
    nav_html = nav_html + media_sub_header

    timestamp = datetime.now(est).strftime("%B %d, %Y %I:%M %p %Z")
    num_matches = len(df_valid)
    stats_bar_full = ANCHOR_PAYLOADS['warning_banner'] + f'<div style="background:#f4f4f4;border-top:1px solid #ddd;border-bottom:1px solid #ddd;font-family:sans-serif;font-size:14px;color:#555;padding:10px 15px;text-align:center;margin-bottom:0;"><strong>Study Data Current As Of:</strong> {timestamp} | <strong>Total Participants:</strong> {num_participants} | <strong>Total Autosomal matches:</strong> {num_matches:,}</div>'
    current_year = str(datetime.now(est).year)
    LEGAL_FOOTER = r'''<div class="legal-footer no-print" style="margin-top:50px;padding:20px;background:#f4f4f4;border-top:1px solid #ddd;text-align:center;color:#666;font-family:sans-serif;font-size:0.85em;clear:both;"><p style="margin-bottom:5px;font-size:1.1em;color:#333;"><strong>&copy; __YEAR__ Ronald Eugene Yates. All Rights Reserved.</strong></p><p style="margin-bottom:5px;">Generated by <em>The Forensic Genealogy Publisher&trade;</em></p><p style="font-style:italic;color:#888;margin-bottom:0;max-width:800px;margin-left:auto;margin-right:auto;">The terms "Forensic Handshake", "Collateral Saturation", and "ANCHOR" are trademarks of Ronald Eugene Yates.</p></div>'''.replace('__YEAR__', current_year)

    REGISTER_SEARCH_BOX = """
    <div class="no-print" style="margin: 20px auto; max-width: 1100px; background: #e0f2f1; padding: 25px; border-radius: 8px; border-left: 6px solid #004d40; box-shadow: 0 4px 15px rgba(0,0,0,0.05); display: flex; flex-direction: column; align-items: center; justify-content: center; font-family: sans-serif;">
        <div style="color: #004d40; font-weight: bold; font-size: 18px; margin-bottom: 15px; text-transform: uppercase; letter-spacing: 1px;">Live Registry Search</div>
        <input type="text" id="register-search" placeholder="🔍 Type a participant name, kit number, ancestor, or cM value to filter..."
               style="width: 100%; max-width: 800px; padding: 15px 25px; border: 2px solid #b2ebf2; border-radius: 30px; font-size: 16px; color: #004d40; outline: none; box-shadow: inset 0 2px 4px rgba(0,0,0,0.05); transition: all 0.3s;"
               onfocus="this.style.borderColor='#004d40'; this.style.boxShadow='0 4px 12px rgba(0,77,64,0.15)';"
               onblur="this.style.borderColor='#b2ebf2'; this.style.boxShadow='inset 0 2px 4px rgba(0,0,0,0.05)';"
               onkeyup="filterRegisterTable()">
    </div>
    <script>
    function filterRegisterTable() {
        let input = document.getElementById('register-search').value.toLowerCase();
        let rows = document.querySelectorAll('table#reg-table tbody tr');
        rows.forEach(row => {
            let text = row.textContent.toLowerCase();
            if(text.includes(input)) {
                row.style.display = '';
            } else {
                row.style.display = 'none';
            }
        });
    }
    </script>
    """

    def render_master(title, content, nav_b, bar, extra_css=""):
        s_info = REGISTER_SEARCH_BOX if nav_b else ""
        content_with_db = content.replace('__DATABASE_SCRIPT__', db_script_tag)
        return master_jinja.render(
            title=title, stats_bar=bar, nav_html=nav_html,
            site_info=s_info, main_content=content_with_db, extra_css=extra_css,
            legal_footer=LEGAL_FOOTER, js_core=SITE_TEMPLATES.get('js_core', ''), btt_btn=SITE_TEMPLATES.get('btt_btn', '')
        )

    def clean_lexicon(raw_html):
        cleaned = raw_html.replace('Biological Proof', 'Biological Validation')
        cleaned = cleaned.replace('biological proof', 'biological validation')
        cleaned = cleaned.replace('Proof-Grade', 'Validation-Grade')
        cleaned = cleaned.replace('proof-grade', 'validation-grade')
        cleaned = cleaned.replace('Integrity Score', 'Validation Status')
        cleaned = cleaned.replace('ranking metric', 'Match Strength')
        cleaned = cleaned.replace('Score Ranges', 'Confidence Levels')
        cleaned = re.sub(r'\bNone\s*\(I(\d+)\)', r'Unnamed Ancestor (I\1)', cleaned)
        return cleaned

    admin_tbody_html = ""
    for r in admin_rows:
        count_style = "color:#c62828; font-weight:bold;" if r['count'] == 0 else "color:#333;"
        admin_tbody_html += f"""<tr data-name="{r['sort']}" data-count="{r['count']}">
            <td style="text-align:left; padding-left:20px;">
                <strong style="color:#00838f; font-size:16px;">{r['name']}</strong> <span style="color:#777; font-size:12px;">(ID: {r['id']})</span><br>
                <span style="font-size:12px; color:#555;">Code: {r['code']} | Token: {r['person_key']}</span>
            </td>
            <td style="font-size:18px; {count_style}">{r['count']}</td>
        </tr>"""

    zm_tbody_html = ""
    if len(zero_rows) == 0: zm_tbody_html = '<tr><td colspan="3" style="text-align:center; padding:40px; color:#2e7d32; font-weight:bold;">All registered participants have matches. No zero-match participants found.</td></tr>'
    else:
        for r in zero_rows:
            zm_tbody_html += f"""<tr>
                <td style="text-align:left; padding-left:20px;"><strong style="color:#b71c1c; font-size:16px;">{r['name']}</strong></td>
                <td style="text-align:left;"><span style="color:#555; font-size:14px;">ID: {r['id']}<br>Code: {r['code']} | Token: {r['person_key']}</span></td>
                <td style="text-align:center; color:#c62828; font-weight:bold; font-size:18px;">0</td>
            </tr>"""

    print("    [+] Mass-Assembling Pages via Jinja2...")

    html_reg_za = df_reg_za.to_html(columns=["Participant Matches & Lineages"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_reg_az = df_reg_az.to_html(columns=["Participant Matches & Lineages"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_reg_match = df_reg_match.to_html(columns=["Participant Matches & Lineages"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")

    html_tree_za = df_tree_za[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_tree_az = df_tree_az[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_tree_match = df_tree_match[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")

    glossary_content = SITE_TEMPLATES.get('glossary_inline', '').replace('<details open ', '<details ').replace('<details open>', '<details>')

    pe_content = SITE_TEMPLATES.get('proof_engine_tmpl', '').replace('__CSS_BASE__', SITE_TEMPLATES.get('css_base', '')).replace('__STATS_BAR__', stats_bar_full).replace('__NAV_HTML__', nav_html).replace('__DATABASE_SCRIPT__', db_script_tag).replace('__LEGAL_FOOTER__', LEGAL_FOOTER)
    pages_to_upload["proof_engine.html"] = pe_content

    dossier_content = SITE_TEMPLATES.get('dossier_tmpl', '').replace('__CSS_BASE__', SITE_TEMPLATES.get('css_base', '')).replace('__STATS_BAR__', stats_bar_full).replace('__NAV_HTML__', nav_html).replace('__DATABASE_SCRIPT__', db_script_tag).replace('__LEGAL_FOOTER__', LEGAL_FOOTER)
    pages_to_upload["dna_dossier.html"] = dossier_content

    admin_content = ANCHOR_PAYLOADS['velvet_rope'] + """
    <style>
    .dash-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px; max-width: 1000px; margin: 30px auto; font-family: sans-serif; }
    .dash-btn { background: white; border: 2px solid #ddd; border-radius: 8px; padding: 25px; text-align: center; text-decoration: none; color: #006064; font-weight: bold; font-size: 16px; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
    .dash-btn:hover { transform: translateY(-3px); box-shadow: 0 6px 12px rgba(0,0,0,0.1); border-color:#00838f; }
    .dash-btn.active-rpt { border-color: #ab47bc; background: #f3e5f5; color: #4a148c; }
    .dash-btn.ged-btn { border-color: #81d4fa; background: #e1f5fe; color: #01579b; }
    .dash-btn.anc-btn { border-color: #ffccbc; background: #fbe9e7; color: #bf360c; }
    .dash-btn.csv-btn { border-color: #ef9a9a; background: #ffebee; color: #c62828; }
    .dash-btn.zm-btn { border-color: #d32f2f; background: #ffebee; color: #b71c1c; }
    .dash-icon { font-size: 32px; display: block; margin-bottom: 10px; }
    .admin-header { border-bottom: 2px solid #004d40; padding-bottom: 15px; color: #004d40; max-width: 1000px; margin: 40px auto 10px auto; font-family: 'Georgia', serif; line-height: 1.6; font-size: 20px; }
    .sort-btns { text-align: center; margin-bottom: 20px; }
    .sort-btn { padding: 10px 20px; margin: 0 5px; border: none; border-radius: 4px; font-weight: bold; cursor: pointer; font-size: 14px; transition: all 0.2s; }
    .sort-btn.active { background: #004d40; color: white; }
    .sort-btn.inactive { background: #e0f2f1; color: #006064; }
    .sort-btn:hover { opacity: 0.9; }
    .admin-table { width: 100%; max-width: 1000px; margin: 0 auto; border-collapse: collapse; font-family: sans-serif; box-shadow: 0 2px 10px rgba(0,0,0,0.05); background: white; }
    .admin-table th { background: #004d40; color: white; padding: 15px; text-align: center; font-size: 15px; }
    .admin-table td { padding: 15px; border-bottom: 1px solid #ddd; text-align: center; vertical-align: middle; }
    .admin-table tr:hover { background-color: #f1f8e9; }
    </style>
    <div class="dash-grid no-print">
        <a href="engine_database.csv?v=__CACHE__" class="dash-btn" style="border-color:#4caf50; background:#e8f5e9; color:#2e7d32;" download><span class="dash-icon">&#128229;</span>Download CSV DB</a>
        <a href="match_to_unmasked.csv?v=__CACHE__" class="dash-btn" style="border-color:#1976d2; background:#e3f2fd; color:#1565c0;" download><span class="dash-icon">&#128193;</span>Download Master CSV</a>
        <a href="dna_report.html" class="dash-btn active-rpt"><span class="dash-icon">&#127891;</span>Report</a>
        <a href="proof_engine.html" class="dash-btn"><span class="dash-icon">&#128300;</span>Proof Engine</a>
        <a href="dna_dossier.html" class="dash-btn"><span class="dash-icon">&#128193;</span>Forensic Dossier</a>
        <a href="gedmatch_hub.html" class="dash-btn ged-btn"><span class="dash-icon">&#129516;</span>GEDmatch Hub</a>
        <a href="anchor_matrix.html" class="dash-btn anc-btn"><span class="dash-icon">&#9875;</span>Anchor Matrix</a>
        <a href="zero_matches.html" class="dash-btn zm-btn"><span class="dash-icon">&#128683;</span>Zero Matches</a>
        <a href="https://analytics.google.com/analytics/web/#/p342112997/reports/intelligenthome" target="_blank" class="dash-btn" style="border-color:#fb8c00; background:#fff3e0; color:#e65100;"><span class="dash-icon">&#128225;</span>Live Telemetry</a>
    </div>
    <h2 class="admin-header">Report - __VALID_COUNT__ Participants with matches</h2>
    <p style="text-align:center; margin-top:5px; margin-bottom:20px; color:#555; font-family:sans-serif;">
        <span style="color:#c62828; font-weight:bold;">__ZERO_COUNT__</span> participants currently report <a href="zero_matches.html" style="color:#c62828; text-decoration:underline;">zero matches</a>.
    </p>
    <div class="sort-btns no-print">
        <button id="btn-sort-az" class="sort-btn active" onclick="sortAdminTable('az')">Sort A-Z (Default)</button>
        <button id="btn-sort-count" class="sort-btn inactive" onclick="sortAdminTable('count')">Sort by Match Count (Low to High)</button>
    </div>
    <div class="table-scroll-wrapper" style="overflow-x:auto;">
        <table class="admin-table">
            <thead><tr><th style="width: 70%; text-align:left; padding-left:20px;">Participant Details</th><th style="width: 30%;">Total Matches</th></tr></thead>
            <tbody id="admin-tbody">
                __ADMIN_TBODY__
            </tbody>
        </table>
    </div>
    __DB_SCRIPT_TAG__
    <script>
    function sortAdminTable(mode) {
        let tbody = document.getElementById('admin-tbody');
        let rows = Array.from(tbody.querySelectorAll('tr'));
        rows.sort((a, b) => {
            if (mode === 'az') return a.getAttribute('data-name').localeCompare(b.getAttribute('data-name'));
            return parseInt(a.getAttribute('data-count')) - parseInt(b.getAttribute('data-count'));
        });
        tbody.innerHTML = '';
        rows.forEach(r => tbody.appendChild(r));
        document.getElementById('btn-sort-az').className = mode === 'az' ? 'sort-btn active' : 'sort-btn inactive';
        document.getElementById('btn-sort-count').className = mode === 'count' ? 'sort-btn active' : 'sort-btn inactive';
    }
    </script>
    """.replace('__CACHE__', str(cache_buster)).replace('__VALID_COUNT__', str(num_participants - len(zero_rows))).replace('__ZERO_COUNT__', str(len(zero_rows))).replace('__ADMIN_TBODY__', admin_tbody_html).replace('__DB_SCRIPT_TAG__', db_script_tag)

    zero_matches_content = ANCHOR_PAYLOADS['velvet_rope'] + """
    <style>
    .zm-header { border-bottom: 2px solid #b71c1c; padding-bottom: 10px; color: #b71c1c; max-width: 1000px; margin: 40px auto 10px auto; font-family: 'Georgia', serif; }
    .zm-subtitle { max-width: 1000px; margin: 0 auto 30px auto; font-family: sans-serif; color: #555; font-size: 15px; line-height: 1.6; }
    .zm-table { width: 100%; max-width: 1000px; margin: 0 auto; border-collapse: collapse; font-family: sans-serif; box-shadow: 0 2px 10px rgba(0,0,0,0.05); background: white; }
    .zm-table th { background: #b71c1c; color: white; padding: 15px; text-align: left; font-size: 15px; }
    .zm-table td { padding: 15px; border-bottom: 1px solid #ddd; text-align: left; vertical-align: middle; }
    .zm-table tr:hover { background-color: #ffebee; }
    </style>
    <div class="no-print" style="max-width: 1000px; margin: 20px auto;">
        <a href="research_admin.html" style="color: #006064; text-decoration: none; font-family: sans-serif; font-weight: bold;">&#8592; Back to Admin Hub</a>
    </div>
    <h2 class="zm-header">Zero Match Audit</h2>
    <p class="zm-subtitle">
        The following participants are registered in the study's authority source but currently report <strong style="color:#c62828;">0</strong> correlated matches in the main engine database.
        <br><span style="display:inline-block; margin-top:10px; padding:5px 10px; background:#ffebee; color:#b71c1c; font-weight:bold; border-radius:4px; border:1px solid #ffcdd2;">Total Zero-Match Participants: __ZERO_COUNT__</span>
    </p>
    <div class="table-scroll-wrapper" style="overflow-x:auto;">
        <table class="zm-table">
            <thead><tr><th style="width: 45%; padding-left:20px;">Participant Name</th><th style="width: 35%;">Participant Details</th><th style="width: 20%; text-align:center;">Match Count</th></tr></thead>
            <tbody>__ZM_TBODY__</tbody>
        </table>
    </div>
    __DB_SCRIPT_TAG__
    """.replace('__ZERO_COUNT__', str(len(zero_rows))).replace('__ZM_TBODY__', zm_tbody_html).replace('__DB_SCRIPT_TAG__', db_script_tag)

    anchor_matrix_content = ANCHOR_PAYLOADS['velvet_rope'] + """
    <style>
    .matrix-table { width: 100%; max-width: 1200px; margin: 0 auto; border-collapse: collapse; font-family: sans-serif; box-shadow: 0 2px 10px rgba(0,0,0,0.05); background: white; }
    .matrix-table th { background: #004d40; color: white; padding: 15px; text-align: center; font-size: 14px; text-transform: uppercase; letter-spacing: 1px; vertical-align: middle; }
    .matrix-table td { padding: 15px; border-bottom: 1px solid #eee; text-align: center; vertical-align: middle; }
    .matrix-table tr:hover { background-color: #f1f8e9; }
    .matrix-table td:first-child { text-align: left; }
    </style>
    <div class="no-print" style="max-width: 1200px; margin: 20px auto;">
        <a href="research_admin.html" style="color: #006064; text-decoration: none; font-family: sans-serif; font-weight: bold;">&#8592; Back to Admin Hub</a>
    </div>
    <div style="max-width:1200px; margin:0 auto; padding:20px; font-family:sans-serif; background:white; border-radius:8px; border-top: 5px solid #004d40; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
        <h2 style="color:#004d40; margin-top:0; display:flex; justify-content:space-between; align-items:center;">
            Forensic Validation Matrix
            <span style="font-size:12px; background:#fff3e0; color:#e65100; padding:4px 10px; border-radius:12px; border:1px solid #ffcc80;">Live Experimentation Mode Active</span>
        </h2>
        <p style="color:#555; font-size:15px; line-height:1.6;">
            This matrix dynamically evaluates each ancestral lineage using the dual-verification <strong>ANCHOR Framework</strong>.
            Adjust the forensic thresholds below to dynamically recalculate network strengths.
        </p>

        <div style="background:#f4f8f9; padding:20px; border-radius:6px; border:1px solid #b2ebf2; margin-bottom:25px; display:flex; gap:20px; align-items:center; justify-content:center; flex-wrap:wrap; font-size:14px; color:#006064;">
            <div style="font-weight:bold; letter-spacing:1px; text-transform:uppercase;">CSS v2a Parameters:</div>
            <label style="display:flex; align-items:center; gap:8px;">Min Proper Matches: <input type="number" id="cfg_pm" style="width:60px; padding:6px; border:1px solid #ccc; border-radius:4px; text-align:center; font-weight:bold;" value="2"></label>
            <label style="display:flex; align-items:center; gap:8px;">Min Vol (cM): <input type="number" id="cfg_vol" style="width:70px; padding:6px; border:1px solid #ccc; border-radius:4px; text-align:center; font-weight:bold;" value="40"></label>
            <label style="display:flex; align-items:center; gap:8px;">Min Collateral Lines: <input type="number" id="cfg_col" style="width:60px; padding:6px; border:1px solid #ccc; border-radius:4px; text-align:center; font-weight:bold;" value="2"></label>
            <button onclick="applyForensicConfig()" style="padding:8px 20px; background:#00838f; color:white; border:none; border-radius:4px; cursor:pointer; font-weight:bold; transition:background 0.2s;">Apply</button>
            <button onclick="resetForensicConfig()" style="padding:8px 20px; background:#cfd8dc; color:#37474f; border:none; border-radius:4px; cursor:pointer; font-weight:bold;">Reset</button>
        </div>

        <div class="table-scroll-wrapper" style="overflow-x:auto;">
            <table class="matrix-table">
                <thead>
                    <tr>
                        <th>Emerged Ancestor</th>
                        <th>Genetic Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(CSS v2a)</span></th>
                        <th>Documentary Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(DOCS)</span></th>
                        <th>ANCHOR Score</th>
                    </tr>
                </thead>
                <tbody id="dynamic-matrix-body">
                </tbody>
            </table>
        </div>
    </div>
    __DB_SCRIPT_TAG__
    <script>
        document.getElementById('cfg_pm').value = localStorage.getItem('ANC_CFG_PM') || 2;
        document.getElementById('cfg_vol').value = localStorage.getItem('ANC_CFG_VOL') || 40;
        document.getElementById('cfg_col').value = localStorage.getItem('ANC_CFG_COL') || 2;

        function applyForensicConfig() {
            localStorage.setItem('ANC_CFG_PM', document.getElementById('cfg_pm').value);
            localStorage.setItem('ANC_CFG_VOL', document.getElementById('cfg_vol').value);
            localStorage.setItem('ANC_CFG_COL', document.getElementById('cfg_col').value);
            renderMatrix();
        }

        function resetForensicConfig() {
            localStorage.setItem('ANC_CFG_PM', 2);
            localStorage.setItem('ANC_CFG_VOL', 40);
            localStorage.setItem('ANC_CFG_COL', 2);
            document.getElementById('cfg_pm').value = 2;
            document.getElementById('cfg_vol').value = 40;
            document.getElementById('cfg_col').value = 2;
            renderMatrix();
        }

        function renderMatrix() {
            if(!window.DB) return;
            let tbody = document.getElementById('dynamic-matrix-body');
            tbody.innerHTML = '<tr><td colspan="4" style="text-align:center; padding:30px; color:#006064; font-weight:bold;">Analyzing Autosomal Network Engine...</td></tr>';

            let t_pm = parseInt(localStorage.getItem('ANC_CFG_PM')) || 2;
            let t_vol = parseFloat(localStorage.getItem('ANC_CFG_VOL')) || 40.0;
            let t_col = parseInt(localStorage.getItem('ANC_CFG_COL')) || 2;

            setTimeout(() => {
                let groups = {};
                window.DB.forEach(r => {
                    let label = r.Dir_Label || r.Authority_Directory_Label;
                    if(!label || label === 'No Matches') return;
                    let cleanLabel = label.split('(')[0].trim();
                    if(!groups[cleanLabel]) groups[cleanLabel] = { pm:0, vol:0, testers: new Set(), lineage: r.Lineage||"" };
                    groups[cleanLabel].pm += 1;
                    groups[cleanLabel].vol += (parseFloat(String(r.cM).replace(/[^0-9.]/g, '')) || 0);
                    groups[cleanLabel].testers.add(r.Participant_Code || r.Tester_Code);
                });

                tbody.innerHTML = '';
                Object.keys(groups).sort().forEach(label => {
                    let g = groups[label];
                    let cssPass = (g.pm >= t_pm && g.vol >= t_vol && g.testers.size >= t_col);
                    let gd = g.lineage ? g.lineage.split('->').length : 0;

                    let cssBadge = cssPass ?
                        "<span style='color:#155724; background:#d4edda; border:1px solid #c3e6cb; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:11px;'>CSS: PASS</span>" :
                        "<span style='color:#721c24; background:#f8d7da; border:1px solid #f5c6cb; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:11px;'>CSS: FAIL</span>";
                    let docsBadge = "<span style='color:#004085; background:#cce5ff; border:1px solid #b8daff; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:11px;'>DOCS: EVAL</span>";

                    let pmPct = Math.round((g.pm / t_pm) * 100);
                    let volPct = Math.round((g.vol / t_vol) * 100);
                    let colPct = Math.round((g.testers.size / t_col) * 100);

                    let cssHtml = `<div style='font-size:13px; color:#444; margin-bottom:8px; display:flex; justify-content:center; gap:8px; flex-wrap:wrap;'>
                        <span style="background:#e8f5e9; padding:4px 8px; border-radius:4px; border:1px solid #c8e6c9;"><b>PM:</b> ${g.pm} <span style="color:#2e7d32; font-size:11px; font-weight:bold;">(${pmPct}%)</span></span>
                        <span style="background:#e8f5e9; padding:4px 8px; border-radius:4px; border:1px solid #c8e6c9;"><b>Vol:</b> ${g.vol.toFixed(1)}cM <span style="color:#2e7d32; font-size:11px; font-weight:bold;">(${volPct}%)</span></span>
                        <span style="background:#e8f5e9; padding:4px 8px; border-radius:4px; border:1px solid #c8e6c9;"><b>Col:</b> ${g.testers.size} <span style="color:#2e7d32; font-size:11px; font-weight:bold;">(${colPct}%)</span></span>
                    </div>${cssBadge}`;

                    let docsHtml = "<div style='font-size:13px; color:#444; margin-bottom:8px;'><b>AX:</b> Yes &nbsp;|&nbsp; <b>GD:</b> " + gd + " &nbsp;|&nbsp; <b>Vts:</b> Pend &nbsp;|&nbsp; <b>CC:</b> Pend &nbsp;|&nbsp; <b>TP:</b> Yes</div>" + docsBadge;

                    let anchorStatus = cssPass ?
                        "<span style='color:white; background:#2e7d32; padding:6px 12px; border-radius:20px; font-weight:bold; font-size:13px; box-shadow:0 2px 4px rgba(0,0,0,0.1);'>🟢 Confirmed</span>" :
                        "<span style='color:white; background:#c62828; padding:6px 12px; border-radius:20px; font-weight:bold; font-size:13px; box-shadow:0 2px 4px rgba(0,0,0,0.1);'>🔴 Speculative</span>";

                    let tr = document.createElement('tr');
                    tr.innerHTML = '<td style="text-align:left;"><strong style="color:#006064; font-size:15px;">' + label + '</strong></td><td>' + cssHtml + '</td><td>' + docsHtml + '</td><td>' + anchorStatus + '</td>';
                    tbody.appendChild(tr);
                });
            }, 100);
        }
        window.addEventListener('load', renderMatrix);
    </script>
    """.replace('__DB_SCRIPT_TAG__', db_script_tag)

    pages_to_upload["research_admin.html"] = render_master("Yates Research Admin Hub", admin_content, False, stats_bar_full, SITE_TEMPLATES.get('admin_css', ''))
    pages_to_upload["zero_matches.html"] = render_master("Zero Matches Audit", zero_matches_content, False, stats_bar_full, "")
    pages_to_upload["anchor_matrix.html"] = render_master("Forensic Validation Matrix", anchor_matrix_content, False, stats_bar_full, "")
    pages_to_upload["about.shtml"] = render_master("Origin of the Methodology", ANCHOR_PAYLOADS['about_content'], False, stats_bar_full, "")

    anchor_injected = SITE_TEMPLATES.get('anchor_content', '').replace('__TAB_FRAMEWORK__', SITE_TEMPLATES.get('anchor_theory', ''))
    anchor_injected = anchor_injected + ANCHOR_PAYLOADS['explanation_html']
    pages_to_upload["anchor.html"] = render_master("ANCHOR Database", anchor_injected, False, stats_bar_full, SITE_TEMPLATES.get('anchor_css', ''))
    pages_to_upload["dna_network.html"] = render_master("DNA Network Visualizer", SITE_TEMPLATES.get('network_page', ''), False, stats_bar_full, "")

    # INJECT SPLITTER ONLY INTO CONSOLIDATOR
    consol_html = SITE_TEMPLATES.get('consolidator_html', '') + SITE_TEMPLATES.get('consolidator_js', '') + ANCHOR_PAYLOADS['report_splitter_js']
    consol_html = consol_html.replace('All Fails Net (Auto-Gen)', 'IGNORED_PRESET')
    pages_to_upload["proof_consolidator.html"] = render_master("Master Proof Report", consol_html, False, stats_bar_full, SITE_TEMPLATES.get('consolidator_css', ''))

    base_style = "text-align:center; margin:15px auto; max-width:1100px; padding:10px; background:#e0f7fa; border:1px solid #b2ebf2; border-radius:4px; font-family:sans-serif; font-size:14px;"
    active_style = "color:#006064; font-weight:bold;"
    link_style = "color:#00acc1; text-decoration:none;"

    toggle_reg_za = f'<div class="no-print" style="{base_style}"><strong>Sort Register:</strong> &nbsp;<span style="{active_style}">By Ancestral Line (Z-A)</span> &nbsp;|&nbsp; <a href="ons_yates_dna_register_participants.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <a href="ons_yates_dna_register_matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_reg_az = f'<div class="no-print" style="{base_style}"><strong>Sort Register:</strong> &nbsp;<a href="ons_yates_dna_register.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <span style="{active_style}">By Participant (A-Z)</span> &nbsp;|&nbsp; <a href="ons_yates_dna_register_matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_reg_match = f'<div class="no-print" style="{base_style}"><strong>Sort Register:</strong> &nbsp;<a href="ons_yates_dna_register.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <a href="ons_yates_dna_register_participants.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <span style="{active_style}">By Match (A-Z)</span></div>'
    toggle_tree_za = f'<div class="no-print" style="{base_style}"><strong>Sort Trees:</strong> &nbsp;<span style="{active_style}">By Ancestral Line (Z-A)</span> &nbsp;|&nbsp; <a href="just-trees-az.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <a href="just-trees-matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_tree_az = f'<div class="no-print" style="{base_style}"><strong>Sort Trees:</strong> &nbsp;<a href="just-trees.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <span style="{active_style}">By Participant (A-Z)</span> &nbsp;|&nbsp; <a href="just-trees-matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_tree_match = f'<div class="no-print" style="{base_style}"><strong>Sort Trees:</strong> &nbsp;<a href="just-trees.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <a href="just-trees-az.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <span style="{active_style}">By Match (A-Z)</span></div>'

    pages_to_upload["ons_yates_dna_register.shtml"] = render_master("ONS Yates Study DNA Register", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_reg_za + f'<div class="table-scroll-wrapper">{html_reg_za}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["ons_yates_dna_register_participants.shtml"] = render_master("ONS Yates Study DNA Register", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_reg_az + f'<div class="table-scroll-wrapper">{html_reg_az}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["ons_yates_dna_register_matches.shtml"] = render_master("ONS Yates Study DNA Register", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_reg_match + f'<div class="table-scroll-wrapper">{html_reg_match}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))

    pages_to_upload["just-trees.shtml"] = render_master("Ancestor Register (Trees View)", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_tree_za + f'<div class="table-scroll-wrapper">{html_tree_za}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["just-trees-az.shtml"] = render_master("Ancestor Register (Trees View)", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_tree_az + f'<div class="table-scroll-wrapper">{html_tree_az}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["just-trees-matches.shtml"] = render_master("Ancestor Register (Trees View)", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_tree_match + f'<div class="table-scroll-wrapper">{html_tree_match}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))

    pages_to_upload["data_glossary.shtml"] = render_master("Data Glossary", glossary_content, False, stats_bar_full, "")
    pages_to_upload["gedmatch_integration.shtml"] = render_master("GEDmatch Hub", SITE_TEMPLATES.get('gedmatch_inline', ''), False, stats_bar_full, "")
    pages_to_upload["contents.shtml"] = render_master("Yates Study User Guide", SITE_TEMPLATES.get('contents_content', ''), False, stats_bar_full, "")
    pages_to_upload["share_dna.shtml"] = render_master("Share Your Ancestry DNA Matches", SITE_TEMPLATES.get('share_content', ''), False, stats_bar_full, "")
    pages_to_upload["subscribe.shtml"] = render_master("Join the Yates Research Community", SITE_TEMPLATES.get('subscribe_content', ''), False, stats_bar_full, "")
    pages_to_upload["dna_theory_of_the_case.htm"] = render_master("Theory of the Case", SITE_TEMPLATES.get('theory_page', ''), False, stats_bar_full, "")

    pages_to_upload["anchor_frame.htm"] = pages_to_upload["anchor.html"]
    pages_to_upload["dna_network.shtml"] = pages_to_upload["dna_network.html"]
    pages_to_upload["admin_singletons.shtml"] = pages_to_upload["ons_yates_dna_register.shtml"]
    pages_to_upload["admin_singletons_participants.shtml"] = pages_to_upload["ons_yates_dna_register_participants.shtml"]
    pages_to_upload["yates_ancestor_register.shtml"] = pages_to_upload["ons_yates_dna_register.shtml"]

    print("    [*] Scanning and updating old header links to the new Anchor directory...")
    for fn in pages_to_upload:
        pages_to_upload[fn] = pages_to_upload[fn].replace("https://yates.one-name.net/ons-study/", "https://yates.one-name.net/anchor/anc-1000-yates/")
        pages_to_upload[fn] = pages_to_upload[fn].replace("/ons-study/", "/anchor/anc-1000-yates/")
        pages_to_upload[fn] = pages_to_upload[fn].replace('href="ons-study/', 'href="anchor/anc-1000-yates/')

    if not os.path.exists('site'): os.makedirs('site')

    print("\n[LOCAL] Overwriting Files & Generating Enhanced Engine Database...")
    df_valid.to_csv(CSV_DB, index=False, encoding="utf-8")
    with open("site/database.js", "w", encoding="utf-8") as f: f.write(js_file_content)

    print("    [✨] Applying final lexicon scrub and injecting UI Upgraders...")
    for p_name, p_content in pages_to_upload.items():
        if p_name.endswith('.html') or p_name.endswith('.shtml') or p_name.endswith('.htm'):
            scrubbed = clean_lexicon(p_content)
            if '</body>' in scrubbed:
                # 🌟 INJECT UNIVERSAL UPGRADER GLOBALLY 🌟
                scrubbed = scrubbed.replace('</body>', f"{ANCHOR_PAYLOADS['deep_path_js']}\n</body>")
            else:
                scrubbed += ANCHOR_PAYLOADS['deep_path_js']
            pages_to_upload[p_name] = scrubbed

    for fn, content in pages_to_upload.items():
        local_path = os.path.join('site', fn)
        if os.path.exists(local_path): os.remove(local_path)
        with open(local_path, "w", encoding="utf-8") as f: f.write(content)

    print("\n[STEP 3] Uploading via FTP to Live Server...")
    upload_success = 0; upload_fails = 0; failed_files = []

    try:
        ftps = FTP_TLS()
        ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()
        ftps.cwd("anchor/anc-1000-yates")

        files_to_upload = list(pages_to_upload.keys()) + ["database.js"]
        if os.path.exists(CSV_DB):
            import shutil
            shutil.copyfile(CSV_DB, os.path.join('site', CSV_DB))
            files_to_upload.append(CSV_DB)

        for i, fn in enumerate(files_to_upload, 1):
            local_path = os.path.join('site', fn)
            try:
                with open(local_path, "rb") as fh: ftps.storbinary(f"STOR {fn}", fh)
                upload_success += 1; print(f"    [{i}/{len(files_to_upload)}] 📤 Success: {fn}")
            except Exception as file_e:
                upload_fails += 1; failed_files.append(fn); print(f"    [{i}/{len(files_to_upload)}] ❌ FAILED: {fn} ({file_e})")

        ftps.quit()
        finish_time = datetime.now(est).strftime("%I:%M:%S %p EST")
        print("\n" + "="*60)
        if upload_fails == 0:
            print(f"✅ PUBLISHER COMPLETE: All {upload_success} files successfully published to London at {finish_time}.")
        else:
            print(f"⚠️ PUBLISHER FINISHED WITH ERRORS: {upload_success} uploaded, {upload_fails} failed at {finish_time}.")
        print("="*60)

    except Exception as e: print(f"\n❌ CRITICAL FTP ERROR: Could not connect to the server. {e}")

run_publisher()

      [CELL 5B] PUBLISHER STARTING (Jinja2 Engine Active)...
    [*] Connecting to FTP Server for Data Sweeps...
    [+] Fetching Central Master Template from London Hub...
    [+] Mass-Assembling Pages via Jinja2...
    [*] Scanning and updating old header links to the new Anchor directory...

[LOCAL] Overwriting Files & Generating Enhanced Engine Database...
    [✨] Applying final lexicon scrub and injecting UI Upgraders...

[STEP 3] Uploading via FTP to Live Server...
    [1/28] 📤 Success: proof_engine.html
    [2/28] 📤 Success: dna_dossier.html
    [3/28] 📤 Success: research_admin.html
    [4/28] 📤 Success: zero_matches.html
    [5/28] 📤 Success: anchor_matrix.html
    [6/28] 📤 Success: about.shtml
    [7/28] 📤 Success: anchor.html
    [8/28] 📤 Success: dna_network.html
    [9/28] 📤 Success: proof_consolidator.html
    [10/28] 📤 Success: ons_yates_dna_register.shtml
    [11/28] 📤 Success: ons_yates_dna_register_participants.shtml
    [12/28] 📤 Success: ons_yates_dna_register_matche

In [25]:
# @title [CELL 6] The Time Machine (Legacy Vault & Rolling 10 Edition)
def run_archiver():
    print("="*60)
    print("      [CELL 6] THE TIME MACHINE (Legacy Vault & Sync)")
    print("="*60)

    import zipfile
    import os
    import pytz
    import json
    from datetime import datetime
    from google.colab import files
    try:
        from google.colab import userdata
    except ImportError:
        pass

    est = pytz.timezone('US/Eastern')
    timestamp = datetime.now(est).strftime("%Y-%m-%d_%H%M_%S")

    target_dir = 'site' if os.path.exists('site') else '.'

    # --- 1. GENERATE THE LEGACY README MANIFESTO ---
    print("[STEP 1] Generating Legacy Handover Protocol...")
    readme_content = """=========================================================
YATES ONS STUDY - LEGACY ENGINE & DATA ARCHIVE
Archive Date: {timestamp}
=========================================================

TO THE FUTURE RESEARCHER:
If you are reading this, you have inherited the complete, self-sustaining
dataset and processing engine for the Yates DNA Study.

This archive does not require access to the original author's cloud accounts,
passwords, or personal computers. Everything required to rebuild, update,
and publish this genealogical network is contained within this folder.

CONTENTS OF THIS VAULT:
1. masked_to_unmasked.csv / match_to_unmasked.csv : The raw authority data and true tester identities.
2. *.ged (GEDCOM) : The master family tree data.
3. site/ (Folder) : The compiled, static HTML website ready for hosting.
4. *.ipynb (The Engine) : The Python source code that processes the data and builds the website.

HOW TO START THE ENGINE:
This system was built to run in a cloud Python environment (specifically Google Colab),
which separates the heavy data processing from the static web hosting.

1. Go to any free Jupyter Notebook environment (like Google Colab: colab.research.google.com).
2. Upload the `.ipynb` file included in this archive to open the Engine.
3. Upload the `.csv` and `.ged` files into the file directory of that environment.
4. Run the Python cells in order from top to bottom.
5. The engine will instantly crunch the data, correlate the DNA nodes, and spit out
   a fresh set of HTML files (the website) which you can then upload to any standard web host.

You hold the keys. The research continues with you.
=========================================================
""".format(timestamp=datetime.now(est).strftime("%B %d, %Y"))

    readme_name = "000_START_HERE_YATES_LEGACY.txt"
    with open(readme_name, "w", encoding="utf-8") as f:
        f.write(readme_content)
    print(f"    ✅ Created Legacy README: {readme_name}")

    # --- 2. GENERATE SITE SNAPSHOT JSON ---
    print("\n[STEP 2] Generating Site Snapshot JSON...")
    snapshot_data = {}
    html_files = [f for f in os.listdir(target_dir) if f.lower().endswith(('.shtml', '.html'))]

    for f_name in html_files:
        try:
            with open(os.path.join(target_dir, f_name), 'r', encoding='utf-8') as fh:
                snapshot_data[f_name] = fh.read()
        except: pass

    snapshot_name = f"site_snapshot_{timestamp}.json"
    with open(snapshot_name, 'w', encoding='utf-8') as f:
        json.dump(snapshot_data, f)
    print(f"    ✅ Created JSON Snapshot: {snapshot_name}")

    # --- 3. CREATE MASTER ZIP VAULT ---
    # NOTICE: Explicitly hunting for GEDCOM (.ged) and Engine (.ipynb) files!
    extensions = ('.csv', '.shtml', '.html', '.json', '.js', '.css', '.ged', '.ipynb')
    files_to_pack = [readme_name]

    for root_dir, dirs, files_in_dir in os.walk('.'):
        if '.config' in root_dir or 'sample_data' in root_dir: continue
        for f in files_in_dir:
            if f.lower().endswith(extensions) and f != snapshot_name:
                files_to_pack.append(os.path.join(root_dir, f))

    zip_name = f"Yates_Study_Legacy_Vault_{timestamp}.zip"

    print(f"\n[STEP 3] Compressing {len(files_to_pack)} files into {zip_name}...")
    try:
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
            for file in files_to_pack:
                arcname = file.replace('./', '') if file.startswith('./') else file
                zf.write(file, arcname)
        print(f"    ✅ Archive Created: {zip_name} ({os.path.getsize(zip_name)/1024/1024:.2f} MB)")
    except Exception as e:
        print(f"    ❌ Compression Failed: {e}")
        return

    # --- 4. FTP UPLOAD & ROLLING 10 CLEANUP ---
    print("\n[STEP 4] Uploading to Web Server Vault (FTP)...")
    try:
        from ftplib import FTP_TLS
        HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
        USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
        PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")

        ftps = FTP_TLS()
        ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()
        ftps.cwd("anchor/anc-1000-yates")

        if "backups" not in ftps.nlst(): ftps.mkd("backups")
        ftps.cwd("backups")

        with open(zip_name, "rb") as fh: ftps.storbinary(f"STOR {zip_name}", fh)
        print(f"    ✅ FTP Success: {zip_name}")

        print("\n    [*] Running Web Vault Cleanup (Rolling 10 Policy)...")
        server_files = ftps.nlst()
        zip_backups = sorted([f for f in server_files if f.startswith("Yates_Study_Legacy_Vault_") and f.endswith(".zip")])

        zips_to_delete = zip_backups[:-10]
        if not zips_to_delete: print("      -> Vault is under 10 backups. No cleanup required.")
        else:
            for old_zip in zips_to_delete:
                try: ftps.delete(old_zip); print(f"      🗑️ Incinerated old FTP ZIP: {old_zip}")
                except: pass
        ftps.quit()
    except Exception as e: print(f"    ⚠️ FTP Upload/Cleanup skipped/failed: {e}")

    # --- 5. DROPBOX SYNC & ROLLING 10 CLEANUP ---
    print("\n[STEP 5] Syncing to Dropbox Omni-Vault...")
    try:
        dbx_app_key = os.environ.get("DBX_APP_KEY") or userdata.get("DBX_APP_KEY")
        dbx_app_secret = os.environ.get("DBX_APP_SECRET") or userdata.get("DBX_APP_SECRET")
        dbx_refresh = os.environ.get("DBX_REFRESH_TOKEN") or userdata.get("DBX_REFRESH_TOKEN")

        if not dbx_refresh: print("    ❌ ERROR: 'DBX_REFRESH_TOKEN' not found.")
        else:
            try: import dropbox
            except ImportError: os.system('pip install dropbox'); import dropbox

            dbx = dropbox.Dropbox(app_key=dbx_app_key, app_secret=dbx_app_secret, oauth2_refresh_token=dbx_refresh)
            folder_path = "/Yates_Study_Sync/archives"

            zip_target = f"{folder_path}/{zip_name}"
            with open(zip_name, "rb") as f: dbx.files_upload(f.read(), zip_target)
            print(f"    ✅ Dropbox Sync Success: {zip_target}")

            print("\n    [*] Running Dropbox Vault Cleanup (Rolling 10 Policy)...")
            try:
                result = dbx.files_list_folder(folder_path)
                dbx_files = [entry.name for entry in result.entries if isinstance(entry, dropbox.files.FileMetadata)]
                dbx_zips = sorted([f for f in dbx_files if f.startswith("Yates_Study_Legacy_Vault_") and f.endswith(".zip")])

                dbx_zips_to_delete = dbx_zips[:-10]
                if not dbx_zips_to_delete: print("      -> Dropbox Vault is under 10 backups. No cleanup required.")
                else:
                    for old_zip in dbx_zips_to_delete:
                        dbx.files_delete_v2(f"{folder_path}/{old_zip}")
                        print(f"      🗑️ Incinerated old Dropbox ZIP: {old_zip}")
            except Exception as sweep_e: print(f"      ⚠️ Dropbox sweeper warning: {sweep_e}")
    except Exception as e: print(f"    ❌ Dropbox Upload Failed: {e}")

    # --- 6. TRIGGER LOCAL DOWNLOAD ---
    print("\n[STEP 6] Triggering Local Download...")
    try: files.download(zip_name); print("    ✅ Please check your browser downloads for the Archive.")
    except: print("    ⚠️ Could not auto-download. You can download manually from the Colab files pane.")

run_archiver()

      [CELL 6] THE TIME MACHINE (Legacy Vault & Sync)
[STEP 1] Generating Legacy Handover Protocol...
    ✅ Created Legacy README: 000_START_HERE_YATES_LEGACY.txt

[STEP 2] Generating Site Snapshot JSON...
    ✅ Created JSON Snapshot: site_snapshot_2026-03-06_1438_04.json

[STEP 3] Compressing 35 files into Yates_Study_Legacy_Vault_2026-03-06_1438_04.zip...
    ✅ Archive Created: Yates_Study_Legacy_Vault_2026-03-06_1438_04.zip (26.19 MB)

[STEP 4] Uploading to Web Server Vault (FTP)...
    ✅ FTP Success: Yates_Study_Legacy_Vault_2026-03-06_1438_04.zip

    [*] Running Web Vault Cleanup (Rolling 10 Policy)...
      🗑️ Incinerated old FTP ZIP: Yates_Study_Legacy_Vault_2026-03-05_2129_54.zip

[STEP 5] Syncing to Dropbox Omni-Vault...
    ✅ Dropbox Sync Success: /Yates_Study_Sync/archives/Yates_Study_Legacy_Vault_2026-03-06_1438_04.zip

    [*] Running Dropbox Vault Cleanup (Rolling 10 Policy)...
      🗑️ Incinerated old Dropbox ZIP: Yates_Study_Legacy_Vault_2026-03-05_2129_54.zip

[STEP 6

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    ✅ Please check your browser downloads for the Archive.
